<a href="https://colab.research.google.com/github/svasanthavada/ladle-optimization-ai/blob/main/notebooks/GA_alloyOptimization_TabTransformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

def load_data(filepath):
    # Use os.path.basename to get the filename for checking the extension
    df = pd.read_excel(filepath, sheet_name="Heats") if os.path.basename(filepath).endswith('.xlsx') else pd.read_csv(filepath)
    df.columns = df.columns.str.strip()
    if df.iloc[0].isnull().sum() < 5:
        df.columns = df.iloc[0]
        df = df[1:]
        df.columns = df.columns.str.strip()
    df.dropna(axis=1, how='all', inplace=True)
    df.dropna(axis=0, how='all', inplace=True)
    return df

def load_summary(filepath):
    # Use os.path.basename to extract filename from filepath
    if os.path.basename(filepath).endswith('.xlsx'): # Use os.path.basename to extract filename
        summary_df = pd.read_excel(filepath, sheet_name="Summary")
        summary_df.columns = summary_df.columns.str.strip()
        summary_df = summary_df.dropna(how='all')
        return summary_df
    return None

def get_target_chemistry(summary_df):
    target_row = summary_df.iloc[0]
    required_elements = ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']
    return {f"F-{el}": float(target_row[el]) for el in required_elements if el in target_row and pd.notnull(target_row[el])}

def create_delta_columns(df):
    open_chem = ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']
    final_chem = [f"F-{el}" for el in ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']]

    for open_col, final_col in zip(open_chem, final_chem):
        if open_col in df.columns and final_col in df.columns:
            delta_col = f"Delta_{open_col.replace('%', '')}"
            df[delta_col] = df[final_col] - df[open_col]
    return df

def handle_missing(df):
    df.fillna(method='ffill', inplace=True)
    df.fillna(df.median(numeric_only=True), inplace=True)
    return df

def clip_outliers(df):
    for col in df.select_dtypes(include=[np.number]).columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        df[col] = df[col].clip(lower=q1 - 1.5 * iqr, upper=q3 + 1.5 * iqr)
    return df

def preprocess_pipeline(filepath):
    df = load_data(filepath)
    summary_df = load_summary(filepath)
    df = handle_missing(df)
    df = create_delta_columns(df)
    df = clip_outliers(df)
    return df, summary_df

def preprocess_data(df):
    df = df.copy()
    # Convert datetime or object to numeric where needed
    for col in df.columns:
        if df[col].dtype == 'O':  # If the column is of object (string) type
            try:
                # Specify the format of your date/time column here
                df[col] = pd.to_datetime(df[col], format='%Y-%m-%d %H:%M:%S').dt.total_seconds()
            except ValueError:  # Handle cases where the column is not a date/time
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    df.fillna(0, inplace=True)

    # Outlier clipping
    for col in df.select_dtypes(include=[np.number]).columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        df[col] = df[col].clip(lower=q1 - 1.5 * iqr, upper=q3 + 1.5 * iqr)

    return df


In [3]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

#from src.preprocessing import load_data, load_summary, preprocess_data

def run_xgboost_regression(filepath):
    # Load and preprocess
    df = load_data(filepath)
    summary_df = load_summary(filepath)
    df = preprocess_data(df)

    # Get target chemistry values from summary
    target_row = summary_df.iloc[0]
    required_elements = ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']
    target_chemistry = {f"F-{el}": float(target_row[el]) for el in required_elements if el in target_row and pd.notnull(target_row[el])}

    # Define full list of alloy and open chemistry input features
    alloy_features = [
        "CSP-SiMn", "Mn HC", "Mn MC", "Mn LC", "Mn Metal", "FeSi", "Ladle Cov",
        "FeMo Metal", "FeV", "FeNb lumps", "FeTi lumps", "FeTi Wire", "FeB", "FeAl",
        "Cal Carb", "Al bar", "Al  wire", "FeP", "Sul Stick", "Al mix", "CaSi wire",
        "Cal Wire", "CaFeAl Wire", "S Wire", "Ni Plate", "FeCr LC", "FeCr HC",
        "Al Shot", "Lead Wire", "Mo Metal", "Syn Slag"
    ]
    open_chemistry = ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']
    process_features = ['Lift Temp', 'Liquidus temp (° C)', 'Arching Time-mm', 'LRF Holding Time-mm', 'LRF Lime']

    input_features = process_features + alloy_features + open_chemistry
    target_columns = list(target_chemistry.keys())

    input_features = [col for col in input_features if col in df.columns]
    target_columns = [col for col in target_columns if col in df.columns]

    if not input_features or not target_columns:
        raise ValueError("Missing required input or target columns in dataset.")

    # Split
    X = df[input_features]
    y = df[target_columns]
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

    # Preprocess
    X_train = preprocess_data(X_train)
    X_val = preprocess_data(X_val)
    X_test = preprocess_data(X_test)
    y_train = preprocess_data(y_train)
    y_val = preprocess_data(y_val)
    y_test = preprocess_data(y_test)

    # Train model
    base_model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)
    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train)

    # Predict and evaluate
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    # Save model
    os.makedirs("models", exist_ok=True)
    model_path = "models/xgboost_multioutput.pkl"
    joblib.dump(model, model_path)

    return model_path, round(rmse, 4), round(r2, 4)

In [4]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

# Assuming load_data, load_summary, and preprocess_data are defined elsewhere in your notebook

def run_xgboost_regression_ranked_alloys(filepath, top_n_alloys=10):
    # Load and preprocess
    df = load_data(filepath)
    summary_df = load_summary(filepath)
    df = preprocess_data(df)

    # Get target chemistry values from summary
    target_row = summary_df.iloc[0]
    required_elements = ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']
    target_chemistry = {f"F-{el}": float(target_row[el]) for el in required_elements if el in target_row and pd.notnull(target_row[el])}

    # Define full list of alloy features
    alloy_features = [
        "CSP-SiMn", "Mn HC", "Mn MC", "Mn LC", "Mn Metal", "FeSi", "Ladle Cov",
        "FeMo Metal", "FeV", "FeNb lumps", "FeTi lumps", "FeTi Wire", "FeB", "FeAl",
        "Cal Carb", "Al bar", "Al  wire", "FeP", "Sul Stick", "Al mix", "CaSi wire",
        "Cal Wire", "CaFeAl Wire", "S Wire", "Ni Plate", "FeCr LC", "FeCr HC",
        "Al Shot", "Lead Wire", "Mo Metal", "Syn Slag"
    ]

    # Calculate alloy usage ranking and select top N
    alloy_usage_counts = (df[alloy_features] != 0).sum().sort_values(ascending=False)
    selected_alloy_features = alloy_usage_counts.head(top_n_alloys).index.tolist()

    # Define process and open chemistry features
    open_chemistry = ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']
    process_features = ['Lift Temp', 'Liquidus temp (° C)', 'Arching Time-mm', 'LRF Holding Time-mm', 'LRF Lime']

    # Combine selected alloy features with process and open chemistry features
    input_features = process_features + selected_alloy_features + open_chemistry
    target_columns = list(target_chemistry.keys())

    # Ensure input features and target columns are present in the DataFrame
    input_features = [col for col in input_features if col in df.columns]
    target_columns = [col for col in target_columns if col in df.columns]

    if not input_features or not target_columns:
        raise ValueError("Missing required input or target columns in dataset.")

    # Split data
    X = df[input_features]
    y = df[target_columns]
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

    # Preprocess
    X_train = preprocess_data(X_train)
    X_val = preprocess_data(X_val)
    X_test = preprocess_data(X_test)
    y_train = preprocess_data(y_train)
    y_val = preprocess_data(y_val)
    y_test = preprocess_data(y_test)


    # Train model
    base_model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)
    model = MultiOutputRegressor(base_model)
    model.fit(X_train, y_train)

    # Predict and evaluate
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    # Save model
    os.makedirs("models", exist_ok=True)
    model_path = "models/xgboost_multioutput_ranked.pkl"
    joblib.dump(model, model_path)

    return model_path, round(rmse, 4), round(r2, 4)

# Example usage:
# filepath = "FE Alloying.xlsx"
# model_path, rmse, r2 = run_xgboost_regression_ranked_alloys(filepath, top_n_alloys=15)
# print(f"Ranked Model saved to: {model_path}")
# print(f"RMSE: {rmse}")
# print(f"R-squared: {r2}")

In [5]:
# Assuming the uploaded file is in the current working directory
filepath = "FE Alloying.xlsx"


model_path, rmse, r2 = run_xgboost_regression(filepath)
print(f"Model saved to: {model_path}")
print(f"RMSE: {rmse}")
print(f"R-squared: {r2}")

model_path, rmse, r2 = run_xgboost_regression_ranked_alloys(filepath)
print(f"Model saved to: {model_path}")
print(f"RMSE: {rmse}")
print(f"R-squared: {r2}")



Model saved to: models/xgboost_multioutput.pkl
RMSE: 0.0034
R-squared: 0.5772
Model saved to: models/xgboost_multioutput_ranked.pkl
RMSE: 0.0034
R-squared: 0.5776


In [6]:
!pip install deap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.6/135.6 kB 2.9 MB/s eta 0:00:00


In [7]:
# ipython-input-22-121fffa05e89 (Corrected)
import numpy as np
import pandas as pd
from deap import base, creator, tools, algorithms
import joblib
from sklearn.metrics import mean_squared_error

# Helper function to mutate within bounds
def mutate_within_bounds(individual, mu, sigma, indpb, bounds):
    """Mutates an individual and clips values to stay within bounds."""
    # Apply standard Gaussian mutation
    tools.mutGaussian(individual, mu, sigma, indpb)
    # Clip mutated values to stay within bounds
    for i, (low, up) in enumerate(bounds):
        individual[i] = np.clip(individual[i], low, up)
    return individual,

# Define the optimization function
def optimize_alloy_additions(model, base_inputs, target_chemistry, alloy_elements, df_successful):
    """
    Optimizes alloy additions using a trained XGBoost model with DEAP,
    handling DEAP setup for multiple calls within the same environment.

    Args:
        model (MultiOutputRegressor): The trained XGBoost multi-output regression model.
        base_inputs (dict): Dictionary of base process and open chemistry inputs.
        target_chemistry (dict): Dictionary of target final chemistry values.
        alloy_elements (list): List of alloy elements to optimize.
        df_successful (pd.DataFrame): DataFrame of successful heats for determining alloy bounds.

    Returns:
        tuple: A dictionary of optimized alloy additions and the predicted final chemistry.
    """
    # Bounds for each alloy based on min/max scaling in successful heats
    # Ensure bounds are non-negative
    # Check if the alloy column exists in df_successful before getting min/max
    bounds = [
        (max(0.0, df_successful[el].min()) if el in df_successful.columns else 0.0,
         df_successful[el].max() if el in df_successful.columns else 100.0) # Provide a default upper bound if column missing
        for el in alloy_elements
    ]
    # Further refine bounds to ensure lower is less than or equal to upper
    bounds = [(low, high) if low <= high else (low, low + 1e-6) for low, high in bounds]


    print("Alloy Bounds:")
    for i, el in enumerate(alloy_elements):
        print(f"{el}: Min = {bounds[i][0]:.2f}, Max = {bounds[i][1]:.4f}")  # Format to 4 decimal places


    # Prepare input for model prediction
    def prepare_input(alloy_values):
      input_dict = base_inputs.copy()
      input_dict.update(dict(zip(alloy_elements, alloy_values)))

      # Ensure the order of columns matches model's expected input
      # Get feature names from the first estimator
      model_feature_names = model.estimators_[0].feature_names_in_
      input_df = pd.DataFrame([input_dict]).reindex(columns=model_feature_names, fill_value=0)

      return input_df

    def fitness_function(individual):
      input_df = prepare_input(individual)
      prediction = model.predict(input_df)[0]

      # Get output feature names (target chemistry elements)
      # Check if feature_names_out_ is available, otherwise use target_chemistry keys
      output_feature_names = getattr(model, 'feature_names_out_', list(target_chemistry.keys()))


      # Create target array in the order of model output features
      # Ensure target has the same order as prediction based on output_feature_names
      # Take target values only for the elements the model predicts (or target_chemistry provides)
      target = np.array([target_chemistry.get(col, 0) for col in output_feature_names])


      # Ensure prediction and target have same number of elements for comparison
      # This handles cases where the model might predict fewer outputs than in target_chemistry
      num_elements_to_compare = min(len(prediction), len(target))
      prediction_subset = prediction[:num_elements_to_compare]
      target_subset = target[:num_elements_to_compare]


      # Calculate RMSE
      return np.sqrt(mean_squared_error(target_subset, prediction_subset)), # Comma for DEAP tuple


    # --- DEAP setup ---
    # Reset creator definitions to allow running multiple times in a notebook
    try:
        del creator.FitnessMin
        del creator.Individual
    except AttributeError:
        pass # Objects not created yet

    # Create new types for this run
    creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
    creator.create("Individual", list, fitness=creator.FitnessMin)

    # Create a new toolbox for this optimization run
    toolbox = base.Toolbox()

    # Register tools for this toolbox instance
    # Modify attr_float to ensure non-negative values
    toolbox.register("attr_float", lambda i: np.random.uniform(bounds[i][0], bounds[i][1]))
    # Modify individual initialization to ensure non-negative values
    toolbox.register("individual", tools.initIterate, creator.Individual,
                     lambda: [np.random.uniform(low, high) for low, high in bounds])
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", fitness_function)
    toolbox.register("mate", tools.cxBlend, alpha=0.5)
    # Register the custom mutation function that handles bounds
    toolbox.register("mutate", mutate_within_bounds, mu=0, sigma=1, indpb=0.2, bounds=bounds)

    toolbox.register("select", tools.selTournament, tournsize=3)


    # Run GA
    pop = toolbox.population(n=30)
    hof = tools.HallOfFame(1)

    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("min", np.min)

    algorithms.eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=20, stats=stats, halloffame=hof, verbose=True)

    # Best solution
    best_individual = hof[0]
    best_prediction = model.predict(prepare_input(best_individual))[0]

    # Clean up DEAP creator definitions explicitly after use if needed,
    # although creating unique names or resetting is safer in notebooks.
    # try:
    #     del creator.FitnessMin
    #     del creator.Individual
    # except AttributeError:
    #     pass

    return dict(zip(alloy_elements, best_individual)), best_prediction

# model = joblib.load("models/xgboost_multioutput.pkl")
# optimized_alloys, predicted_chem = optimize_alloy_additions(model, base_inputs, target_chem, alloy_list, df_successful)

In [13]:
import numpy as np
import pandas as pd
from deap import base, creator, tools, algorithms
import joblib
from sklearn.metrics import mean_squared_error

# Helper function to mutate within bounds (can be shared if identical logic)
# Redefined here for clarity in this code block, but could be defined once globally
def mutate_within_bounds(individual, mu, sigma, indpb, bounds):
    """Mutates an individual and clips values to stay within bounds."""
    # Apply standard Gaussian mutation
    tools.mutGaussian(individual, mu, sigma, indpb)
    # Clip mutated values to stay within bounds
    for i, (low, up) in enumerate(bounds):
        individual[i] = np.clip(individual[i], low, up)
    return individual,


def optimize_alloy_additions_ranked(model_path, base_inputs, target_chemistry, all_alloy_elements, df_successful):
    """
    Optimizes alloy additions using a ranked XGBoost model, considering only the alloys
    that the model was trained on. Handles DEAP setup for multiple calls.

    Args:
        model_path (str): Path to the trained ranked XGBoost model.
        base_inputs (dict): Dictionary of base process and open chemistry inputs.
        target_chemistry (dict): Dictionary of target final chemistry values.
        all_alloy_elements (list): List of all possible alloy elements.
        df_successful (pd.DataFrame): DataFrame of successful heats for determining alloy bounds.

    Returns:
        tuple: A dictionary of optimized alloy additions (including all alloys,
               with ignored ones set to 0) and the predicted final chemistry.
    """
    # Load the ranked model
    model = joblib.load(model_path)

    # Get the feature names the ranked model was trained on
    # Access the first estimator to get feature names
    ranked_model_features = model.estimators_[0].feature_names_in_


    # Identify the alloy features the ranked model used
    ranked_alloy_elements = [col for col in ranked_model_features if col in all_alloy_elements]

    # Identify the alloys that are NOT in the ranked set
    ignored_alloy_elements = [col for col in all_alloy_elements if col not in ranked_alloy_elements]

    # Bounds for the ranked alloys based on min/max scaling in successful heats
    # Ensure bounds are non-negative
    # Check if the alloy column exists in df_successful before getting min/max
    bounds = [
        (max(0.0, df_successful[el].min()) if el in df_successful.columns else 0.0,
         df_successful[el].max() if el in df_successful.columns else 100.0) # Provide a default upper bound if column missing
        for el in ranked_alloy_elements
    ]
    # Further refine bounds to ensure lower is less than or equal to upper
    bounds = [(low, high) if low <= high else (low, low + 1e-6) for low, high in bounds]


    print("Ranked Alloy Bounds:")
    for i, el in enumerate(ranked_alloy_elements):
        print(f"{el}: Min = {bounds[i][0]:.2f}, Max = {bounds[i][1]:.4f}")

    # Prepare input for model prediction (includes both ranked and ignored alloys)
    def prepare_input(ranked_alloy_values):
        input_dict = base_inputs.copy()

        # Update with optimized values for ranked alloys
        input_dict.update(dict(zip(ranked_alloy_elements, ranked_alloy_values)))

        # Set ignored alloys to 0
        input_dict.update(dict(zip(ignored_alloy_elements, [0.0] * len(ignored_alloy_elements))))

        # Ensure the order of columns matches model's expected input
        # Use ranked_model_features which is the exact order the model expects
        input_df = pd.DataFrame([input_dict]).reindex(columns=ranked_model_features, fill_value=0)

        return input_df

    def fitness_function(individual):
        input_df = prepare_input(individual)
        prediction = model.predict(input_df)[0]

        # Get output feature names (target chemistry elements)
        # Check if feature_names_out_ is available (scikit-learn >= 1.2.0), use it
        # Otherwise, use the keys from target_chemistry which should align with model output
        output_feature_names = getattr(model, 'feature_names_out_', list(target_chemistry.keys()))


        # Create target array in the order of model output features
        # Ensure target has the same order as prediction based on output_feature_names
        target = np.array([target_chemistry.get(col, 0) for col in output_feature_names])

        # Ensure prediction and target have same number of elements for comparison
        # This handles cases where the model might predict fewer outputs than in target_chemistry
        num_elements_to_compare = min(len(prediction), len(target))
        prediction_subset = prediction[:num_elements_to_compare]
        target_subset = target[:num_elements_to_compare]


        # Calculate RMSE
        return np.sqrt(mean_squared_error(target_subset, prediction_subset)), # Comma for DEAP tuple

    # --- DEAP setup ---
    # Reset creator definitions to allow running multiple times in a notebook
    try:
        del creator.FitnessMinRanked
        del creator.IndividualRanked
    except AttributeError:
        pass # Objects not created yet

    # Create new types for this run
    creator.create("FitnessMinRanked", base.Fitness, weights=(-1.0,))
    creator.create("IndividualRanked", list, fitness=creator.FitnessMinRanked)

    # Create a new toolbox for this optimization run
    toolbox = base.Toolbox()

    # Register tools for this toolbox instance
    # Unregistering loop is removed as a new toolbox is created
    toolbox.register("attr_float", lambda i: np.random.uniform(bounds[i][0], bounds[i][1]))
    toolbox.register("individual", tools.initIterate, creator.IndividualRanked,
                     lambda: [np.random.uniform(low, high) for low, high in bounds])
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", fitness_function)
    toolbox.register("mate", tools.cxBlend, alpha=0.5)
    # Register the custom mutation function that handles bounds
    toolbox.register("mutate", mutate_within_bounds, mu=0, sigma=1, indpb=0.2, bounds=bounds)

    toolbox.register("select", tools.selTournament, tournsize=3)

    # Run GA
    pop = toolbox.population(n=30)
    hof = tools.HallOfFame(1) # Corrected typo: HallofFame -> HallOfFame

    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("min", np.min)

    # Corrected variable names in algorithms.eaSimple call
    algorithms.eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=20, stats=stats, halloffame=hof, verbose=True)

    # Best solution (only for ranked alloys)
    best_individual_ranked = hof[0]

    # Create the full optimized alloys dictionary, including ignored ones as 0
    optimized_alloys_full = dict(zip(ranked_alloy_elements, best_individual_ranked))
    optimized_alloys_full.update(dict(zip(ignored_alloy_elements, [0.0] * len(ignored_alloy_elements))))

    # Predict using the full set of optimized alloys
    # Ensure the input format for prediction matches the model's expectation
    best_prediction_input_df = prepare_input(best_individual_ranked)
    best_prediction = model.predict(best_prediction_input_df)[0]

    # Clean up DEAP creator definitions explicitly after use if needed,
    # although creating unique names or resetting is safer in notebooks.
    # try:
    #     del creator.FitnessMinRanked
    #     del creator.IndividualRanked
    # except AttributeError:
    #     pass

    return optimized_alloys_full, best_prediction

In [11]:
# Corrected and Completed Code for optimize_alloy_additions_ranked modified
import numpy as np
import pandas as pd
from deap import base, creator, tools, algorithms
import joblib
from sklearn.metrics import mean_squared_error
import random

# Helper function to mutate within bounds (shared)
def mutate_within_bounds(individual, mu, sigma, indpb, bounds):
    """Mutates an individual and clips values to stay within bounds."""
    tools.mutGaussian(individual, mu, sigma, indpb)
    for i, (low, up) in enumerate(bounds):
        individual[i] = np.clip(individual[i], low, up)
    return individual,


def optimize_alloy_additions_ranked(model_path, base_inputs, target_chemistry, all_alloy_elements, df_successful):
    """
    Optimizes alloy additions using a ranked XGBoost model, considering only the alloys
    that the model was trained on. Collects all valid (non-negative) solutions
    during the optimization and selects the best one based on fitness.

    Args:
        model_path (str): Path to the trained ranked XGBoost model.
        base_inputs (dict): Dictionary of base process and open chemistry inputs.
        target_chemistry (dict): Dictionary of target final chemistry values.
        all_alloy_elements (list): List of all possible alloy elements.
        df_successful (pd.DataFrame): DataFrame of successful heats for determining alloy bounds.

    Returns:
        tuple: A dictionary of optimized alloy additions (including all alloys,
               with ignored ones set to 0) and the predicted final chemistry.
    """
    # Load the ranked model
    model = joblib.load(model_path)

    # Get the feature names the ranked model was trained on
    ranked_model_features = model.estimators_[0].feature_names_in_
    ranked_alloy_elements = [col for col in ranked_model_features if col in all_alloy_elements]
    ignored_alloy_elements = [col for col in all_alloy_elements if col not in ranked_alloy_elements]

    # Bounds for the ranked alloys
    bounds = [
        (max(0.0, df_successful[el].min()) if el in df_successful.columns else 0.0,
         df_successful[el].max() if el in df_successful.columns else 100.0)
        for el in ranked_alloy_elements
    ]
    bounds = [(low, high) if low <= high else (low, low + 1e-6) for low, high in bounds]

    print("Ranked Alloy Bounds:")
    for i, el in enumerate(ranked_alloy_elements):
        print(f"{el}: Min = {bounds[i][0]:.4f}, Max = {bounds[i][1]:.4f}")

    # Prepare input for model prediction
    def prepare_input(ranked_alloy_values):
        input_dict = base_inputs.copy()
        input_dict.update(dict(zip(ranked_alloy_elements, ranked_alloy_values)))
        input_dict.update(dict(zip(ignored_alloy_elements, [0.0] * len(ignored_alloy_elements))))
        input_df = pd.DataFrame([input_dict]).reindex(columns=ranked_model_features, fill_value=0)
        return input_df

    def fitness_function(individual):
        input_df = prepare_input(individual)
        prediction = model.predict(input_df)[0]
        output_feature_names = getattr(model, 'feature_names_out_', list(target_chemistry.keys()))
        target = np.array([target_chemistry.get(col, 0) for col in output_feature_names])
        num_elements_to_compare = min(len(prediction), len(target))
        prediction_subset = prediction[:num_elements_to_compare]
        target_subset = target[:num_elements_to_compare]
        return np.sqrt(mean_squared_error(target_subset, prediction_subset)), # Comma for DEAP tuple

    # --- DEAP setup ---
    try:
        del creator.FitnessMinRanked
        del creator.IndividualRanked
    except AttributeError:
        pass

    creator.create("FitnessMinRanked", base.Fitness, weights=(-1.0,))
    creator.create("IndividualRanked", list, fitness=creator.FitnessMinRanked)

    toolbox = base.Toolbox()
    toolbox.register("attr_float", lambda i: np.random.uniform(bounds[i][0], bounds[i][1]))
    toolbox.register("individual", tools.initIterate, creator.IndividualRanked,
                     lambda: [np.random.uniform(low, high) for low, high in bounds])
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", fitness_function)
    toolbox.register("mate", tools.cxBlend, alpha=0.5)
    toolbox.register("mutate", mutate_within_bounds, mu=0, sigma=1, indpb=0.3, bounds=bounds)
    toolbox.register("select", tools.selTournament, tournsize=3)

    # --- Optimization ---
    pop = toolbox.population(n=100) # Increased population size
    hof = tools.HallOfFame(1)

    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("min", np.min)

    # Store history of valid individuals and their fitness
    valid_solutions_history = []

    # Run GA and collect valid solutions
    for gen in range(50): # Run for 50 generations
        offspring = toolbox.select(pop, len(pop))
        offspring = list(toolbox.clone(ind) for ind in offspring)

        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < 0.5:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if random.random() < 0.3:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        # Evaluate the individuals with an invalid fitness and store valid ones
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit

        # Store valid (non-negative) individuals from the current generation
        for ind in offspring:
            is_valid = all(value >= 0 for value in ind)
            if is_valid:
                 # Ensure values are within bounds before storing (redundant if clipping works, but safe)
                 clipped_ind = [np.clip(ind[i], bounds[i][0], bounds[i][1]) for i in range(len(ind))]
                 valid_solutions_history.append({"individual": clipped_ind, "fitness": ind.fitness.values[0]})


        # Update the hall of fame with the generated individuals
        hof.update(offspring)

        # Replace the current population by the offspring
        pop[:] = offspring

        # Gather statistics (optional)
        # record = stats.compile(pop)
        # print(f"Gen {gen}: {record}")

    print("Optimization finished.")

    # --- Select the Best Valid Solution from History ---

    if not valid_solutions_history:
        print("Warning: No valid (non-negative) solutions found during optimization.")
        # If no valid solutions were found, fall back to the best solution from Hall of Fame
        # which might still contain negative values, or handle as appropriate for your application.
        print("Returning best found by Hall of Fame (may contain negative values).")
        best_individual_ranked = hof[0]
    else:
        # Sort valid solutions by fitness (RMSE), ascending
        valid_solutions_history.sort(key=lambda x: x["fitness"])

        # Select the best valid solution based on lowest RMSE
        best_valid_solution = valid_solutions_history[0]
        best_individual_ranked = best_valid_solution["individual"]
        print(f"Selected best valid solution from history with RMSE: {best_valid_solution['fitness']:.4f}")


    # Create the full optimized alloys dictionary, including ignored ones as 0
    optimized_alloys_full = dict(zip(ranked_alloy_elements, best_individual_ranked))
    optimized_alloys_full.update(dict(zip(ignored_alloy_elements, [0.0] * len(ignored_alloy_elements))))

    # Predict using the full set of optimized alloys
    best_prediction_input_df = prepare_input(best_individual_ranked)
    best_prediction = model.predict(best_prediction_input_df)[0]

    return optimized_alloys_full, best_prediction

In [14]:
# --- Data Loading and Preprocessing ---
# Preprocess
df, summary_df = preprocess_pipeline(filepath)

# Define the list of basic chemistry elements
# This list is used in multiple places (feature definitions, target extraction)
chemistry_cols = ['C', 'Mn', 'S', 'P', 'Si', 'Cr', 'Ni', 'Mo', 'V', 'Ti', 'Al', 'Ca', 'N', 'Pb', 'Nb']


# Extract alloy and process features
alloy_cols = [
    "CSP-SiMn", "Mn HC", "Mn MC", "Mn LC", "Mn Metal", "FeSi", "Ladle Cov",
    "FeMo Metal", "FeV", "FeNb lumps", "FeTi lumps", "FeTi Wire", "FeB", "FeAl",
    "Cal Carb", "Al bar", "Al  wire", "FeP", "Sul Stick", "Al mix", "CaSi wire",
    "Cal Wire", "CaFeAl Wire", "S Wire", "Ni Plate", "FeCr LC", "FeCr HC",
    "Al Shot", "Lead Wire", "Mo Metal", "Syn Slag"
]

# Define process and open chemistry features (using chemistry_cols)
process_chem_cols = [
    'Lift Temp', 'Liquidus temp (° C)', 'Arching Time-mm',
    'LRF Holding Time-mm', 'LRF Lime'
] + [f'{el}%' for el in chemistry_cols]


# Extract successful heats for defining optimization bounds
max_summary_row = summary_df.iloc[2].fillna(0)
aim_summary_row = summary_df.iloc[3].fillna(0)
min_summary_row = summary_df.iloc[1].fillna(0)

target_keys = [k.strip() for k in max_summary_row.index if k.strip().endswith("%")]
target_vector = max_summary_row[target_keys].dropna().astype(float)
present_f_cols = [f"F-{k}" for k in target_keys if f"F-{k}" in df.columns]

# Calculate Success Score based on deviation from max target
# You can adjust this score calculation method based on your definition of "successful"
df['Success_Score'] = -((df[present_f_cols] - target_vector.loc[[k for k in target_keys if f"F-{k}" in df.columns]].values) ** 2).sum(axis=1) ** 0.3

# Select heats with a high success score (e.g., top 50%)
df_successful = df[df['Success_Score'] >= df['Success_Score'].quantile(0.5)].copy() # Use .copy() to avoid SettingWithCopyWarning

print(f"Shape of df_successful: {df_successful.shape}")
# print("Description of alloy columns in df_successful:")
# print(df_successful[alloy_cols].describe()) # Uncomment to see describe output


# Select base inputs from median of successful heats
selected_cols_for_base = [col for col in process_chem_cols + alloy_cols if col in df_successful.columns]
base_inputs = df_successful[selected_cols_for_base].median(numeric_only=True).to_dict()

# Extract target chemistry dictionaries (using chemistry_cols for consistency)
max_chem = {
    f"F-{el}%": float(max_summary_row.get(f"{el}%", np.nan)) for el in chemistry_cols
    if pd.notnull(max_summary_row.get(f"{el}%"))
}

aim_chem = {
    f"F-{el}%": float(aim_summary_row.get(f"{el}%", np.nan)) for el in chemistry_cols
    if pd.notnull(aim_summary_row.get(f"{el}%"))
}

min_chem = {
    f"F-{el}%": float(min_summary_row.get(f"{el}%", np.nan)) for el in chemistry_cols
    if pd.notnull(min_summary_row.get(f"{el}%"))
}

# --- Train and Load Models ---
# Run and save the full XGBoost model
# (Assuming run_xgboost_regression function uses the defined process_chem_cols and alloy_cols)
model_path_full, rmse_full, r2_full = run_xgboost_regression(filepath)
print(f"\nFull Model saved to: {model_path_full}")
print(f"RMSE (Full): {rmse_full}")
print(f"R-squared (Full): {r2_full}")

# Run and save the ranked XGBoost model (adjust top_n_alloys as needed)
# (Assuming run_xgboost_regression_ranked_alloys function uses the defined process_chem_cols and alloy_cols)
ranked_model_path, rmse_ranked, r2_ranked = run_xgboost_regression_ranked_alloys(filepath, top_n_alloys=20)
print(f"\nRanked Model saved to: {ranked_model_path}")
print(f"RMSE (Ranked): {rmse_ranked}")
print(f"R-squared (Ranked): {r2_ranked}")

# Load the trained models
model_full = joblib.load(model_path_full)
model_ranked = joblib.load(ranked_model_path)

# Ensure base_inputs have the same columns as the training data for each model
# Full model
expected_features_full = model_full.estimators_[0].feature_names_in_
base_inputs_full_df = pd.DataFrame([base_inputs])
base_inputs_full_df = base_inputs_full_df.reindex(columns=expected_features_full, fill_value=0)
base_inputs_full = base_inputs_full_df.iloc[0].to_dict()

# Ranked model
expected_features_ranked = model_ranked.estimators_[0].feature_names_in_
base_inputs_ranked_df = pd.DataFrame([base_inputs])
base_inputs_ranked_df = base_inputs_ranked_df.reindex(columns=expected_features_ranked, fill_value=0)
base_inputs_ranked = base_inputs_ranked_df.iloc[0].to_dict()


# --- Run Optimization for Both Models ---

print("\n--- Running Optimization with Full XGBoost Model ---")
# Assuming optimize_alloy_additions is defined earlier in the notebook
optimized_alloys_full, predicted_chem_full = optimize_alloy_additions(
    model=model_full,
    base_inputs=base_inputs_full,
    target_chemistry=max_chem, # Using max_chem as target
    alloy_elements=alloy_cols, # Optimize over all possible alloys
    df_successful=df_successful
)

print("\n--- Running Optimization with Ranked XGBoost Model ---")
# Assuming optimize_alloy_additions_ranked is defined earlier in the notebook
optimized_alloys_ranked, predicted_chem_ranked = optimize_alloy_additions_ranked(
    model_path=ranked_model_path, # Pass the path to the ranked model
    base_inputs=base_inputs_ranked,
    target_chemistry=max_chem, # Using max_chem as target
    all_alloy_elements=alloy_cols, # Pass the full list of alloys
    df_successful=df
)

# ipython-input-41-58e5940901a0 (Corrected)

# Assuming all preceding code (data loading, preprocessing, model training/loading,
# optimization functions, and chemistry_cols definition) has been executed.

# --- Compare Results ---

print("\n--- Comparison of Optimized Alloy Additions ---")
# Assuming optimized_alloys_full and optimized_alloys_ranked are available from previous steps
optimized_alloys_comparison = pd.DataFrame({
    "Full Model (Recommended kg)": optimized_alloys_full,
    "Ranked Model (Recommended kg)": optimized_alloys_ranked
})
# Sort by total recommended amount or importance for better comparison
# For now, just display
print(optimized_alloys_comparison)


print("\n--- Comparison of Predicted Final Chemistry vs Target ---")

# Ensure keys are consistent for comparison DataFrame using the defined chemistry_cols
# This combines keys from max_chem, aim_chem, min_chem, and constructs keys from chemistry_cols
all_chem_keys = sorted(list(set(max_chem.keys()) | set(aim_chem.keys()) | set(min_chem.keys()) |
                            set([f"F-{el}%" for el in chemistry_cols]))) # Include all possible F-chem keys

comparison_data = {
    "Min Target": [min_chem.get(k, np.nan) for k in all_chem_keys],
    "Aim Target": [aim_chem.get(k, np.nan) for k in all_chem_keys],
    "Max Target": [max_chem.get(k, np.nan) for k in all_chem_keys],
}

# Align predicted chemistry with the comparison keys
# Use the keys from max_chem (or any of the target chem dicts) as the output feature names
# since MultiOutputRegressor predicts these directly.
# Ensure predicted_chem_full and predicted_chem_ranked arrays are in the same order
# as the columns they were trained on (which should align with max_chem keys).
# A safer way is to align based on the target columns used during training.
# Let's assume the order of predictions aligns with the order of keys in max_chem
# because max_chem was used to define the target_columns in run_xgboost_regression.

# Retrieve the actual target column names used during training from the model structure
# MultiOutputRegressor stores the original target columns as part of its fit process
# in the target_names_ or similar attribute. Let's check the structure.
# Based on typical MultiOutputRegressor behavior with pandas DataFrames,
# the columns of the training target DataFrame y_train will be used.
# We can use the keys from max_chem, assuming they match the order and names
# of the target columns the models were trained on. This is consistent with the training code.

output_feature_names = list(max_chem.keys()) # Assuming max_chem keys are the target column names in order

# Create dictionaries from predictions using these output feature names
predicted_full_dict = dict(zip(output_feature_names, predicted_chem_full))
predicted_ranked_dict = dict(zip(output_feature_names, predicted_chem_ranked))


comparison_data["Predicted (Full Model)"] = [predicted_full_dict.get(k, np.nan) for k in all_chem_keys]
comparison_data["Predicted (Ranked Model)"] = [predicted_ranked_dict.get(k, np.nan) for k in all_chem_keys]


chem_comparison_df = pd.DataFrame(comparison_data, index=all_chem_keys)

# Add Delta columns for easier comparison to Aim
chem_comparison_df['Delta to Aim (Full)'] = chem_comparison_df['Predicted (Full Model)'] - chem_comparison_df['Aim Target']
chem_comparison_df['Delta to Aim (Ranked)'] = chem_comparison_df['Predicted (Ranked Model)'] - chem_comparison_df['Aim Target']


print(chem_comparison_df.round(5))

# Optional: Calculate and print RMSE for the predicted vs Aim chemistry for each model's optimization result
# This gives an idea of how close the optimization got to the Aim target.
# Ensure you only compare elements present in the Aim target.
# Use the output_feature_names to align the predicted values for RMSE calculation
aim_keys_present_in_predictions = [k for k in aim_chem.keys() if k in output_feature_names]

if aim_keys_present_in_predictions:
    aim_values = np.array([aim_chem[k] for k in aim_keys_present_in_predictions])
    predicted_full_values = np.array([predicted_full_dict[k] for k in aim_keys_present_in_predictions])
    predicted_ranked_values = np.array([predicted_ranked_dict[k] for k in aim_keys_present_in_predictions])


    rmse_predicted_full_to_aim = np.sqrt(mean_squared_error(aim_values, predicted_full_values))
    rmse_predicted_ranked_to_aim = np.sqrt(mean_squared_error(aim_values, predicted_ranked_values))

    print(f"\nRMSE of Predicted vs Aim (Full Model Optimization): {rmse_predicted_full_to_aim:.4f}")
    print(f"RMSE of Predicted vs Aim (Ranked Model Optimization): {rmse_predicted_ranked_to_aim:.4f}")
else:
    print("\nCannot calculate RMSE of Predicted vs Aim: No common chemistry elements between Aim and model predictions.")

<ipython-input-2-85496d986d3d>:41: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
<ipython-input-2-85496d986d3d>:41: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(method='ffill', inplace=True)


Shape of df_successful: (1621, 89)

Full Model saved to: models/xgboost_multioutput.pkl
RMSE (Full): 0.0034
R-squared (Full): 0.5772

Ranked Model saved to: models/xgboost_multioutput_ranked.pkl
RMSE (Ranked): 0.0034
R-squared (Ranked): 0.5776

--- Running Optimization with Full XGBoost Model ---
Alloy Bounds:
CSP-SiMn: Min = 0.00, Max = 0.0000
Mn HC: Min = 0.00, Max = 80.0000
Mn MC: Min = 0.00, Max = 0.0000
Mn LC: Min = 0.00, Max = 0.0000
Mn Metal: Min = 0.00, Max = 0.0000
FeSi: Min = 0.00, Max = 0.0000
Ladle Cov: Min = 0.00, Max = 0.0000
FeMo Metal: Min = 0.00, Max = 0.0000
FeV: Min = 0.00, Max = 0.0000
FeNb lumps: Min = 0.00, Max = 0.0000
FeTi lumps: Min = 0.00, Max = 0.0000
FeTi Wire: Min = 0.00, Max = 0.0000
FeB: Min = 0.00, Max = 0.0000
FeAl: Min = 0.00, Max = 0.0000
Cal Carb: Min = 0.00, Max = 0.0000
Al bar: Min = 0.00, Max = 0.0000
Al  wire: Min = 0.00, Max = 264.5500
FeP: Min = 0.00, Max = 0.0000
Sul Stick: Min = 0.00, Max = 0.0000
Al mix: Min = 0.00, Max = 0.0000
CaSi wire: M

In [ ]:
import pandas as pd
import numpy as np

def sensitivity_analysis(model, base_inputs, optimized_alloys, target_chem, alloy_elements, perturbation_percentage=0.1):
    results = []

    for element in alloy_elements:
        # Perturb the optimized value of the current element
        perturbation = optimized_alloys[element] * perturbation_percentage

        # Create input with the perturbed element value
        perturbed_inputs = base_inputs.copy()
        perturbed_inputs.update(optimized_alloys)  # Add optimized alloys to base inputs
        perturbed_inputs[element] += perturbation

        # Ensure order of columns matches model's expected input
        input_df = pd.DataFrame([perturbed_inputs], columns=model.estimators_[0].feature_names_in_)

        # Make predictions with the perturbed inputs
        predicted_chem = model.predict(input_df)[0]

        # Calculate the change in predicted final chemistry compared to target
        # Ensure both predicted_chem and target_chem have the same length
        num_elements_to_compare = min(len(predicted_chem), len(target_chem))

        change_from_target = {
            f"Delta_{k}": predicted_chem[i] - target_chem.get(k, 0)  # Handle missing target values with 0
            for i, k in enumerate(target_chem.keys()) if i < num_elements_to_compare # Limit iterations to the common elements
        }

        results.append({
            "Element": element,
            "Perturbation": perturbation,
            "Predicted Final Chemistry": predicted_chem,
            **change_from_target
        })

    return pd.DataFrame(results)

In [ ]:
!pip install scikit-learn --upgrade

In [ ]:
!pip install scikit-optimize

In [ ]:
!pip install scikit-learn --upgrade
import numpy as np
import pandas as pd
from skopt import gp_minimize
import joblib  # Make sure to import joblib

# ... (Your existing code for loading data, preprocessing, etc.)

# Load your trained model
model = joblib.load("models/xgboost_multioutput_ranked.pkl")

# Define the objective function for Bayesian Optimization
def objective_function(alloy_values):
    # Prepare input for the model
    input_dict = base_inputs.copy()
    input_dict.update(dict(zip(alloy_cols, alloy_values)))  # Use alloy_cols here

    # Ensure the order of columns matches model's expected input
    input_df = pd.DataFrame([input_dict], columns=model.estimators_[0].feature_names_in_)

    # Make predictions
    prediction = model.predict(input_df)[0]

    # Calculate the objective (e.g., RMSE)
    # Check if feature_names_out_ is available, otherwise use target_chem keys
    output_feature_names = getattr(model, 'feature_names_out_', list(max_chem.keys()))
    target = np.array([max_chem.get(col, 0) for col in output_feature_names[:len(prediction)]])  # Use max_chem
    rmse = np.sqrt(np.mean((prediction - target) ** 2))
    return rmse

# Define the search space (bounds for alloy elements)
space = [(df[el].min(), df[el].max()) for el in alloy_cols]  # Use alloy_cols
#space = [(df_successful[el].min(), df_successful[el].max()) for el in alloy_cols]  # Use alloy_cols

# Check and adjust bounds
space = [(lower, upper) if lower < upper else (lower, lower + 1e-6) for lower, upper in space]

# Perform Bayesian Optimization
result = gp_minimize(objective_function, space, n_calls=50, random_state=42)

# Get the optimal alloy additions
optimized_alloys_bayesian = dict(zip(alloy_cols, result.x))  # Use alloy_cols

# Print the optimized alloy additions
print("Optimized Alloy Additions (Bayesian Optimization):")
print(optimized_alloys_bayesian)



Optimized Alloy Additions (Bayesian Optimization):
{'CSP-SiMn': 2.3570679874656453e-07, 'Mn HC': np.int64(80), 'Mn MC': 5.578985042419153e-08, 'Mn LC': 9.97071698438963e-07, 'Mn Metal': 9.085971846698589e-07, 'FeSi': 9.80230463275513e-07, 'Ladle Cov': 5.494228078225672e-07, 'FeMo Metal': 7.43505850275698e-07, 'FeV': 0.0, 'FeNb lumps': 2.371713109804601e-07, 'FeTi lumps': 3.154433135158834e-07, 'FeTi Wire': 9.629484580445636e-07, 'FeB': 3.432117221885347e-08, 'FeAl': 5.091649913946369e-07, 'Cal Carb': 9.538264588667747e-07, 'Al bar': 3.0987336146093005e-07, 'Al  wire': 0.0, 'FeP': 7.322483884137368e-07, 'Sul Stick': 7.187703747436823e-08, 'Al mix': 5.896261237340135e-07, 'CaSi wire': 1.8404561840903292e-07, 'Cal Wire': 25.249999999999996, 'CaFeAl Wire': 1e-06, 'S Wire': 0.0, 'Ni Plate': 6.200505934291418e-08, 'FeCr LC': 2.8309516270166834e-07, 'FeCr HC': 4.016335084860689e-08, 'Al Shot': 1.2727176896370526e-07, 'Lead Wire': 3.378631471712096e-07, 'Mo Metal': 0.0, 'Syn Slag': 3.896029657

In [ ]:
importances = model.estimators_[0].feature_importances_
feature_names = model.estimators_[0].feature_names_in_
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
print(feature_importance_df.sort_values(by='Importance', ascending=False))

                Feature  Importance
25                   C%    0.572178
27                   S%    0.034243
7                 Mn HC    0.030750
26                  Mn%    0.027909
2       Arching Time-mm    0.026884
31                  Ni%    0.025460
29                  Si%    0.024112
37                   N%    0.023544
6              Al  wire    0.021347
4              LRF Lime    0.021333
39                  Nb%    0.021136
30                  Cr%    0.021113
32                  Mo%    0.020743
36                  Ca%    0.020733
38                  Pb%    0.020013
28                   P%    0.019542
5              Cal Wire    0.018767
35                  Al%    0.018712
34                  Ti%    0.018245
0             Lift Temp    0.013237
1   Liquidus temp (° C)    0.000000
3   LRF Holding Time-mm    0.000000
8                 Mn LC    0.000000
9              Mn Metal    0.000000
23               Al bar    0.000000
22                  FeP    0.000000
21                 FeAl    0

In [ ]:
df = load_data("FE Alloying.xlsx")
df_summary = load_summary(filepath)
df_heats = preprocess_data(df)

# Re-define chemistry columns
chemistry_cols = ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']
final_chem_cols = [f"F-{el}" for el in chemistry_cols]

# Retry LP execution
sample_row = df_heats.dropna(subset=chemistry_cols + final_chem_cols).iloc[100]
opening_chem = {el: float(sample_row[el]) for el in chemistry_cols}
df_summary.columns = df_summary.columns.str.strip()
df_summary_indexed = df_summary.set_index('Steel Grade > G-600-A')
target_row = df_summary_indexed.loc['Aim'].dropna()
target_chem = {k: float(v) for k, v in target_row.items() if isinstance(v, (float, int))}

# Calculate needed delta per element
delta_chem = {
    k.strip('%'): max(0, target_chem.get(k, 0) - opening_chem.get(k, 0))
    for k in target_chem if k in opening_chem
}

# Rebuild LP matrix
selected_alloys = ["Mn HC", "Al  wire", "Cal Wire"]
alloy_contributions = {
    "Mn HC": {"Mn": 0.75},
    "Al  wire": {"Al": 0.99},
    "Cal Wire": {"Ca": 0.31}
}

elements = list({el for contrib in alloy_contributions.values() for el in contrib})
b_vector = np.array([delta_chem.get(el, 0) for el in elements])

A_ub = -np.array([
    [alloy_contributions[alloy].get(el, 0) for alloy in selected_alloys]
    for el in elements
])
b_ub = -b_vector

c = np.ones(len(selected_alloys))
bounds = [(0, 200)] * len(selected_alloys)

from scipy.optimize import linprog
lp_result = linprog(c=c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
success = lp_result.success
optimized_alloys = dict(zip(selected_alloys, lp_result.x)) if success else {}
(success, lp_result.fun, optimized_alloys)


(True,
 0.020239999999999998,
 {'Mn HC': np.float64(0.020239999999999998),
  'Al  wire': np.float64(0.0),
  'Cal Wire': np.float64(0.0)})

In [ ]:
# Prepare input dictionary for prediction using optimized alloy additions
base_inputs = {
    'Lift Temp': float(sample_row['Lift Temp']),
    'Liquidus temp (° C)': float(sample_row['Liquidus temp (° C)']),
    'Arching Time-mm': float(sample_row['Arching Time-mm']),
    'LRF Holding Time-mm': float(sample_row.get('LRF Holding Time-mm', 0)),
    'LRF Lime': float(sample_row.get('LRF Lime', 0)),
    'C%': float(sample_row['C%']),
    'Mn%': float(sample_row['Mn%']),
    'S%': float(sample_row['S%']),
    'P%': float(sample_row['P%']),
    'Si%': float(sample_row['Si%']),
    'Cr%': float(sample_row['Cr%']),
    'Ni%': float(sample_row['Ni%']),
    'Mo%': float(sample_row['Mo%']),
    'V%': float(sample_row['V%']),
    'Ti%': float(sample_row['Ti%']),
    'Al%': float(sample_row['Al%']),
    'Ca%': float(sample_row['Ca%']),
    'N%': float(sample_row['N%']),
    'Pb%': float(sample_row['Pb%']),
    'Nb%': float(sample_row['Nb%']),
    'Mn HC': optimized_alloys.get('Mn HC', 0.0),
    'Al  wire': optimized_alloys.get('Al  wire', 0.0),
    'Cal Wire': optimized_alloys.get('Cal Wire', 0.0),
}

model = joblib.load("models/xgboost_multioutput.pkl")

# Ensure all expected features exist
feature_order = list(model.estimators_[0].feature_names_in_)
input_row = pd.DataFrame([{col: base_inputs.get(col, 0.0) for col in feature_order}])

# Run XGBoost multi-output prediction
predicted_chem = model.predict(input_row)[0]

# Extract target AIM values for those same final chemistry outputs
predicted_columns = [f"F-{el}%" for el in chemistry_cols if f"F-{el}%" in df_summary_indexed.columns]
aim_targets = df_summary_indexed.loc["Aim", predicted_columns].astype(float).values

# Compare predicted vs AIM values
comparison_df = pd.DataFrame({
    "Target (Aim)": aim_targets,
    "Predicted": predicted_chem[:len(aim_targets)]
}, index=predicted_columns)

comparison_df.round(5)


,Target (Aim),Predicted


In [ ]:
!pip install pyswarms

In [ ]:
import numpy as np
import pandas as pd
from pyswarms.single.global_best import GlobalBestPSO  # Import PSO from pyswarms
import joblib

# Define the optimization function (using PSO)
def run_pso_optimization(model, base_inputs, target_chemistry, alloy_elements, df_successful):
    # Bounds for each alloy based on min/max scaling in successful heats
    bounds = [(df_successful[el].min(), df_successful[el].max()) for el in alloy_elements]

    # Display bounds inline (using f-strings for formatting)
    print("Alloy Bounds:")
    for i, el in enumerate(alloy_elements):
        print(f"{el}: Min = {bounds[i][0]:.2f}, Max = {bounds[i][1]:.4f}")  # Format to 4 decimal places

    # Normalize input features for model prediction
    def prepare_input(alloy_values):
        input_dict = base_inputs.copy()
        input_dict.update(dict(zip(alloy_elements, alloy_values)))

        # Ensure the order of columns matches model's expected input
        input_df = pd.DataFrame([input_dict], columns=model.estimators_[0].feature_names_in_)

        return input_df

    # Fitness function for PSO
    def fitness_function(alloy_matrix):
        predictions = []
        for row in alloy_matrix:  # Iterate through particles
            input_df = prepare_input(row)
            prediction = model.predict(input_df)[0]

            # ***CHANGE***: Get output feature names from the model
            # (or use target_chemistry keys if feature_names_out_ is not available)
            output_feature_names = getattr(model.estimators_[0], 'feature_names_out_', list(target_chemistry.keys()))

            # ***CHANGE***: Filter target_chemistry to only include elements predicted by the model
            filtered_target_chemistry = {k: v for k, v in target_chemistry.items() if k in output_feature_names}

            # Create target array in the order of model output features
            target = np.array([filtered_target_chemistry.get(col, 0) for col in output_feature_names[:len(prediction)]])

            # Ensure prediction and target have the same number of elements
            num_elements_to_compare = min(len(prediction), len(target))
            prediction = prediction[:num_elements_to_compare]
            target = target[:num_elements_to_compare]  # Slice target as well

            # Calculate loss (RMSE)
            loss = np.sqrt(np.mean((prediction - target) ** 2))
            predictions.append(loss)

        return np.array(predictions)  # Return fitness values for all particles

    # Define the bounds and number of particles
    lower_bounds = [bound[0] for bound in bounds]
    upper_bounds = [bound[1] for bound in bounds]
    n_particles = 50  # Adjust as needed

    # Initialize and run PSO
    options = {'c1': 0.5, 'c2': 0.7, 'w': 0.4}  # Adjust parameters as needed
    optimizer = GlobalBestPSO(n_particles=n_particles, dimensions=len(alloy_elements),
                            options=options, bounds=(lower_bounds, upper_bounds))
    cost, pos = optimizer.optimize(fitness_function, iters=100)  # Adjust iterations as needed

    # Best solution and prediction
    best_individual = pos  # The best particle's position (alloy values)
    best_prediction = model.predict(prepare_input(best_individual))[0]

    # Return the optimized alloys and predicted chemistry
    return dict(zip(alloy_elements, best_individual)), best_prediction

In [ ]:
# Preprocess
df, summary_df = preprocess_pipeline(filepath)

# Extract alloy and process features
alloy_cols = [
    "CSP-SiMn", "Mn HC", "Mn MC", "Mn LC", "Mn Metal", "FeSi", "Ladle Cov",
    "FeMo Metal", "FeV", "FeNb lumps", "FeTi lumps", "FeTi Wire", "FeB", "FeAl",
    "Cal Carb", "Al bar", "Al  wire", "FeP", "Sul Stick", "Al mix", "CaSi wire",
    "Cal Wire", "CaFeAl Wire", "S Wire", "Ni Plate", "FeCr LC", "FeCr HC",
    "Al Shot", "Lead Wire", "Mo Metal", "Syn Slag"
]

process_chem_cols = [
    'Lift Temp', 'Liquidus temp (° C)', 'Arching Time-mm',
    'LRF Holding Time-mm', 'LRF Lime',
    'C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%'
]

# Extract successful heats
max_summary_row = summary_df.iloc[2].fillna(0)
aim_summary_row = summary_df.iloc[3].fillna(0)
min_summary_row = summary_df.iloc[1].fillna(0)
print(aim_summary_row)
target_keys = [k.strip() for k in max_summary_row.index if k.strip().endswith("%")]
target_vector = max_summary_row[target_keys].dropna().astype(float)
present_f_cols = [f"F-{k}" for k in target_keys if f"F-{k}" in df.columns]
df['Success_Score'] = -((df[present_f_cols] - target_vector.loc[[k for k in target_keys if f"F-{k}" in df.columns]].values) ** 2).sum(axis=1) ** 0.3
df_successful = df[df['Success_Score'] >= df['Success_Score'].quantile(0.5)]


# Get target chemical keys (elements with %)
# target_keys = [k.strip() for k in max_summary_row.index if k.strip().endswith("%")]

# # Create a mask for rows within min-max range for each F-chemical
# mask = pd.Series(True, index=df.index)  # Initialize mask to True for all rows

# 1. Attempt filtering with min-max range
# mask = pd.Series(True, index=df.index)
# for element in target_keys:
#     f_col = f"F-{element}"
#     if f_col in df.columns:
#         min_val = min_summary_row.get(element, -np.inf)
#         max_val = max_summary_row.get(element, np.inf)
#         mask &= (df[f_col] >= min_val) & (df[f_col] <= max_val)

# df_successful = df[mask]

# # 2. If df_successful is empty, consider aim closer values
# if df_successful.empty:
#     df['Success_Score'] = 0  # Initialize Success_Score column
#     for element in target_keys:
#         f_col = f"F-{element}"
#         if f_col in df.columns:
#             aim_val = aim_summary_row.get(element, 0)  # Get aim value
#             df['Success_Score'] += (df[f_col] - aim_val) ** 2  # Calculate squared difference

#     df['Success_Score'] = df['Success_Score'] ** 0.5  # Take square root for overall distance
#     df_successful = df[df['Success_Score'] <= df['Success_Score'].quantile(0.5)]  # Select closest 50%


print(df_successful.shape)  # Check if it's empty
print(df_successful[alloy_cols].describe())  # Check summary statistics

# Select base inputs from median of successful heats
selected_cols = [col for col in process_chem_cols + alloy_cols if col in df_successful.columns]
base_inputs = df_successful[selected_cols].median(numeric_only=True).to_dict()

# Extract target chemistry
max_chem = {
    f"F-{k.strip()}": float(v) for k, v in max_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v)
}

aim_chem = {
    f"F-{k.strip()}": float(v) for k, v in aim_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v)
}

min_chem = {
    f"F-{k.strip()}": float(v) for k, v in min_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v)
}

# Load model
model = joblib.load("models/xgboost_multioutput.pkl")

print("Model Features:")
print(model.estimators_[0].feature_names_in_)

# Ensure base_inputs have the same columns as the training data
expected_features = model.estimators_[0].feature_names_in_
base_inputs_df = pd.DataFrame([base_inputs])  # Convert to DataFrame
base_inputs_df = base_inputs_df.reindex(columns=expected_features, fill_value=0)  # Reindex and fill missing

# Convert back to dictionary
base_inputs = base_inputs_df.iloc[0].to_dict()

optimized_alloys, predicted_chem = run_pso_optimization(
    model=model,
    base_inputs=base_inputs,
    target_chemistry=max_chem,
    alloy_elements=alloy_cols,
    df_successful=df_successful
)

# Display results
print("Optimized Alloy Additions")
pd.DataFrame(optimized_alloys, index=["Recommended (kg)"])
optimized_alloys


print("Predicted Final Chemistry vs Target")
chem_df = pd.DataFrame({
    "Min": min_chem,
    "Target": max_chem,
    "Aim": aim_chem,
    "Predicted": dict(zip(aim_chem.keys(), predicted_chem))  # Use aim_chem.keys()
})

chem_df

#st.success("Optimization completed! Review suggested alloy additions above.")!pip install pyswarms

<ipython-input-77-85496d986d3d>:41: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
<ipython-input-77-85496d986d3d>:41: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(method='ffill', inplace=True)


Unnamed: 0                   0
Unnamed: 1                   0
Steel Grade > G-600-A      Aim
C%                        0.05
Mn%                       0.12
S%                         0.0
P%                       0.015
Si%                       0.04
Cr%                        0.0
Ni%                        0.0
Mo%                        0.0
V%                         0.0
Ti%                        0.0
Al%                      0.038
Ca%                        0.0
N%                         0.0
Pb%                        0.0
Nb%                        0.0
B%                         0.0
CU%                        0.0
Name: 3, dtype: object
(1621, 89)
0      CSP-SiMn        Mn HC   Mn MC   Mn LC  Mn Metal    FeSi  Ladle Cov  \
count    1621.0  1621.000000  1621.0  1621.0    1621.0  1621.0     1621.0   
mean        0.0    15.586058     0.0     0.0       0.0     0.0        0.0   
std         0.0    23.905463     0.0     0.0       0.0     0.0        0.0   
min         0.0     0.000000     0.0  

2025-05-18 12:49:28,111 - pyswarms.single.global_best - INFO - Optimize for 100 iters with {'c1': 0.5, 'c2': 0.7, 'w': 0.4}


Model Features:
['Lift Temp' 'Liquidus temp (° C)' 'Arching Time-mm' 'LRF Holding Time-mm'
 'LRF Lime' 'CSP-SiMn' 'Mn HC' 'Mn MC' 'Mn LC' 'Mn Metal' 'FeSi'
 'Ladle Cov' 'FeMo Metal' 'FeV' 'FeNb lumps' 'FeTi lumps' 'FeTi Wire'
 'FeB' 'FeAl' 'Cal Carb' 'Al bar' 'Al  wire' 'FeP' 'Sul Stick' 'Al mix'
 'CaSi wire' 'Cal Wire' 'CaFeAl Wire' 'S Wire' 'Ni Plate' 'FeCr LC'
 'FeCr HC' 'Al Shot' 'Lead Wire' 'Mo Metal' 'Syn Slag' 'C%' 'Mn%' 'S%'
 'P%' 'Si%' 'Cr%' 'Ni%' 'Mo%' 'V%' 'Ti%' 'Al%' 'Ca%' 'N%' 'Pb%' 'Nb%']
Alloy Bounds:
CSP-SiMn: Min = 0.00, Max = 0.0000
Mn HC: Min = 0.00, Max = 80.0000
Mn MC: Min = 0.00, Max = 0.0000
Mn LC: Min = 0.00, Max = 0.0000
Mn Metal: Min = 0.00, Max = 0.0000
FeSi: Min = 0.00, Max = 0.0000
Ladle Cov: Min = 0.00, Max = 0.0000
FeMo Metal: Min = 0.00, Max = 0.0000
FeV: Min = 0.00, Max = 0.0000
FeNb lumps: Min = 0.00, Max = 0.0000
FeTi lumps: Min = 0.00, Max = 0.0000
FeTi Wire: Min = 0.00, Max = 0.0000
FeB: Min = 0.00, Max = 0.0000
FeAl: Min = 0.00, Max = 0.0000
Cal Ca

pyswarms.single.global_best:   0%|          |0/100, best_cost=0.0184/usr/local/lib/python3.11/dist-packages/pyswarms/backend/handlers.py:387: RuntimeWarning: invalid value encountered in remainder
  new_pos[greater_than_bound] = lb[greater_than_bound] + np.mod(
pyswarms.single.global_best: 100%|██████████|100/100, best_cost=0.0183
2025-05-18 13:07:13,924 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.018274676624864934, best pos: [        nan 60.13505466         nan         nan         nan         nan
         nan         nan         nan         nan         nan         nan
         nan         nan         nan         nan 87.78873975         nan
         nan         nan         nan 25.2070519          nan         nan
         nan         nan         nan         nan         nan         nan
         nan]


Optimized Alloy Additions
Predicted Final Chemistry vs Target


,Min,Target,Aim,Predicted
F-C%,0.02,0.0600,0.050,0.042450
F-Mn%,0.10,0.1500,0.120,0.138519
F-S%,0.00,0.0100,0.000,0.007395
F-P%,0.00,0.0300,0.015,0.013852
F-Si%,0.00,0.0500,0.040,0.025043
F-Cr%,0.00,0.0500,0.000,0.013933
F-Ni%,0.00,0.0500,0.000,0.003472
F-Mo%,0.00,0.0050,0.000,0.000400
F-V%,0.00,0.0030,0.000,0.000200
F-Ti%,0.00,0.0030,0.000,0.000941


In [ ]:
optimized_alloys

{'CSP-SiMn': np.float64(nan),
 'Mn HC': np.float64(60.13505465972843),
 'Mn MC': np.float64(nan),
 'Mn LC': np.float64(nan),
 'Mn Metal': np.float64(nan),
 'FeSi': np.float64(nan),
 'Ladle Cov': np.float64(nan),
 'FeMo Metal': np.float64(nan),
 'FeV': np.float64(nan),
 'FeNb lumps': np.float64(nan),
 'FeTi lumps': np.float64(nan),
 'FeTi Wire': np.float64(nan),
 'FeB': np.float64(nan),
 'FeAl': np.float64(nan),
 'Cal Carb': np.float64(nan),
 'Al bar': np.float64(nan),
 'Al  wire': np.float64(87.78873974568569),
 'FeP': np.float64(nan),
 'Sul Stick': np.float64(nan),
 'Al mix': np.float64(nan),
 'CaSi wire': np.float64(nan),
 'Cal Wire': np.float64(25.207051900571425),
 'CaFeAl Wire': np.float64(nan),
 'S Wire': np.float64(nan),
 'Ni Plate': np.float64(nan),
 'FeCr LC': np.float64(nan),
 'FeCr HC': np.float64(nan),
 'Al Shot': np.float64(nan),
 'Lead Wire': np.float64(nan),
 'Mo Metal': np.float64(nan),
 'Syn Slag': np.float64(nan)}

In [ ]:
import numpy as np
import pandas as pd
import random
import joblib

# Load your trained model
model = joblib.load("models/xgboost_multioutput.pkl")

# Preprocess (Assuming preprocess_pipeline is defined elsewhere)
df, summary_df = preprocess_pipeline(filepath)


# Extract alloy and process features
alloy_cols = [
    "CSP-SiMn", "Mn HC", "Mn MC", "Mn LC", "Mn Metal", "FeSi", "Ladle Cov",
    "FeMo Metal", "FeV", "FeNb lumps", "FeTi lumps", "FeTi Wire", "FeB", "FeAl",
    "Cal Carb", "Al bar", "Al  wire", "FeP", "Sul Stick", "Al mix", "CaSi wire",
    "Cal Wire", "CaFeAl Wire", "S Wire", "Ni Plate", "FeCr LC", "FeCr HC",
    "Al Shot", "Lead Wire", "Mo Metal", "Syn Slag"
]

process_chem_cols = [
    'Lift Temp', 'Liquidus temp (° C)', 'Arching Time-mm',
    'LRF Holding Time-mm', 'LRF Lime',
    'C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%'
]

# Extract successful heats
max_summary_row = summary_df.iloc[2].fillna(0)
aim_summary_row = summary_df.iloc[3].fillna(0)
min_summary_row = summary_df.iloc[1].fillna(0)
print(aim_summary_row)
target_keys = [k.strip() for k in max_summary_row.index if k.strip().endswith("%")]
target_vector = max_summary_row[target_keys].dropna().astype(float)
present_f_cols = [f"F-{k}" for k in target_keys if f"F-{k}" in df.columns]
df['Success_Score'] = -((df[present_f_cols] - target_vector.loc[[k for k in target_keys if f"F-{k}" in df.columns]].values) ** 2).sum(axis=1) ** 0.3
df_successful = df[df['Success_Score'] >= df['Success_Score'].quantile(0.5)]


# Get target chemical keys (elements with %)
# target_keys = [k.strip() for k in max_summary_row.index if k.strip().endswith("%")]

# # Create a mask for rows within min-max range for each F-chemical
# mask = pd.Series(True, index=df.index)  # Initialize mask to True for all rows

# 1. Attempt filtering with min-max range
# mask = pd.Series(True, index=df.index)
# for element in target_keys:
#     f_col = f"F-{element}"
#     if f_col in df.columns:
#         min_val = min_summary_row.get(element, -np.inf)
#         max_val = max_summary_row.get(element, np.inf)
#         mask &= (df[f_col] >= min_val) & (df[f_col] <= max_val)

# df_successful = df[mask]

# # 2. If df_successful is empty, consider aim closer values
# if df_successful.empty:
#     df['Success_Score'] = 0  # Initialize Success_Score column
#     for element in target_keys:
#         f_col = f"F-{element}"
#         if f_col in df.columns:
#             aim_val = aim_summary_row.get(element, 0)  # Get aim value
#             df['Success_Score'] += (df[f_col] - aim_val) ** 2  # Calculate squared difference

#     df['Success_Score'] = df['Success_Score'] ** 0.5  # Take square root for overall distance
#     df_successful = df[df['Success_Score'] <= df['Success_Score'].quantile(0.5)]  # Select closest 50%


print(df_successful.shape)  # Check if it's empty
print(df_successful[alloy_cols].describe())  # Check summary statistics

# Select base inputs from median of successful heats
selected_cols = [col for col in process_chem_cols + alloy_cols if col in df_successful.columns]
base_inputs = df_successful[selected_cols].median(numeric_only=True).to_dict()

# Extract target chemistry
max_chem = {
    f"F-{k.strip()}": float(v) for k, v in max_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v)
}

aim_chem = {
    f"F-{k.strip()}": float(v) for k, v in aim_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v)
}

min_chem = {
    f"F-{k.strip()}": float(v) for k, v in min_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v)
}

# Load model
model = joblib.load("models/xgboost_multioutput.pkl")

print("Model Features:")
print(model.estimators_[0].feature_names_in_)

# Ensure base_inputs have the same columns as the training data
expected_features = model.estimators_[0].feature_names_in_
base_inputs_df = pd.DataFrame([base_inputs])  # Convert to DataFrame
base_inputs_df = base_inputs_df.reindex(columns=expected_features, fill_value=0)  # Reindex and fill missing

# Convert back to dictionary
base_inputs = base_inputs_df.iloc[0].to_dict()
# Define the objective function for Simulated Annealing
def objective_function(alloy_values):
    input_dict = base_inputs.copy()
    input_dict.update(dict(zip(alloy_cols, alloy_values)))

    # Ensure the order of columns matches model's expected input
    input_df = pd.DataFrame([input_dict], columns=model.estimators_[0].feature_names_in_)

    prediction = model.predict(input_df)[0]

    # Calculate the objective (RMSE)
    output_feature_names = getattr(model, 'feature_names_out_', list(max_chem.keys()))  # Get output feature names
    target = np.array([max_chem.get(col, 0) for col in output_feature_names[:len(prediction)]]) # Align model prediction with target
    rmse = np.sqrt(np.mean((prediction - target) ** 2))
    return rmse

# Define the search space (bounds for alloy elements)
# Use df_successful for bounds to match GA and PSO
space = [(df_successful[el].min(), df_successful[el].max()) for el in alloy_cols]

# Check and adjust bounds (if necessary)
space = [(lower, upper) if lower < upper else (lower, lower + 1e-6) for lower, upper in space]

# Define the Simulated Annealing function
def simulated_annealing(objective_function, space, initial_temp=1000, cooling_rate=0.95, num_iterations=1000):
    current_solution = [random.uniform(lower, upper) for lower, upper in space]
    current_temp = initial_temp
    best_solution = current_solution
    best_objective = objective_function(current_solution)
    current_objective = best_objective  # Initialize current_objective

    for _ in range(num_iterations):
        neighbor_solution = current_solution[:]
        index = random.randint(0, len(space) - 1)
        neighbor_solution[index] += random.uniform(-1, 1) * (space[index][1] - space[index][0]) * 0.1
        neighbor_solution[index] = np.clip(neighbor_solution[index], space[index][0], space[index][1])

        neighbor_objective = objective_function(neighbor_solution)

        if neighbor_objective < current_objective or random.random() < np.exp(-(neighbor_objective - current_objective) / current_temp):
            current_solution = neighbor_solution
            current_objective = neighbor_objective
            if current_objective < best_objective:
                best_solution = current_solution
                best_objective = current_objective

        current_temp *= cooling_rate

    return best_solution, best_objective

# Run Simulated Annealing
best_solution, best_objective = simulated_annealing(objective_function, space)

# Get the optimal alloy additions
optimized_alloys_sa = dict(zip(alloy_cols, best_solution))

# Display results
print("Optimized Alloy Additions (Simulated Annealing):")
print(optimized_alloys_sa)

print("Predicted Final Chemistry vs Target")
chem_df = pd.DataFrame({
    "Min": min_chem,
    "Target": max_chem,
    "Aim": aim_chem,
    "Predicted": dict(zip(aim_chem.keys(), predicted_chem))  # Use aim_chem.keys()
})

chem_df

<ipython-input-77-85496d986d3d>:41: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
<ipython-input-77-85496d986d3d>:41: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(method='ffill', inplace=True)


Unnamed: 0                   0
Unnamed: 1                   0
Steel Grade > G-600-A      Aim
C%                        0.05
Mn%                       0.12
S%                         0.0
P%                       0.015
Si%                       0.04
Cr%                        0.0
Ni%                        0.0
Mo%                        0.0
V%                         0.0
Ti%                        0.0
Al%                      0.038
Ca%                        0.0
N%                         0.0
Pb%                        0.0
Nb%                        0.0
B%                         0.0
CU%                        0.0
Name: 3, dtype: object
(1621, 89)
0      CSP-SiMn        Mn HC   Mn MC   Mn LC  Mn Metal    FeSi  Ladle Cov  \
count    1621.0  1621.000000  1621.0  1621.0    1621.0  1621.0     1621.0   
mean        0.0    15.586058     0.0     0.0       0.0     0.0        0.0   
std         0.0    23.905463     0.0     0.0       0.0     0.0        0.0   
min         0.0     0.000000     0.0  

,Min,Target,Aim,Predicted
F-C%,0.02,0.0600,0.050,0.042450
F-Mn%,0.10,0.1500,0.120,0.138519
F-S%,0.00,0.0100,0.000,0.007395
F-P%,0.00,0.0300,0.015,0.013852
F-Si%,0.00,0.0500,0.040,0.025043
F-Cr%,0.00,0.0500,0.000,0.013933
F-Ni%,0.00,0.0500,0.000,0.003472
F-Mo%,0.00,0.0050,0.000,0.000400
F-V%,0.00,0.0030,0.000,0.000200
F-Ti%,0.00,0.0030,0.000,0.000941


In [127]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from deap import base, creator, tools, algorithms
import random
import joblib
import os
import datetime
from IPython.display import display



def load_data(filepath):
    # Use os.path.basename to get the filename for checking the extension
    df = pd.read_excel(filepath, sheet_name="Heats") if os.path.basename(filepath).endswith('.xlsx') else pd.read_csv(filepath)
    df.columns = df.columns.str.strip()
    if df.iloc[0].isnull().sum() < 5:
        df.columns = df.iloc[0]
        df = df[1:]
        df.columns = df.columns.str.strip()
    df.dropna(axis=1, how='all', inplace=True)
    df.dropna(axis=0, how='all', inplace=True)
    return df

def load_summary(filepath):
    # Use os.path.basename to extract filename from filepath
    if os.path.basename(filepath).endswith('.xlsx'): # Use os.path.basename to extract filename
        summary_df = pd.read_excel(filepath, sheet_name="Summary")
        summary_df.columns = summary_df.columns.str.strip()
        summary_df = summary_df.dropna(how='all')
        return summary_df
    return None

def handle_missing(df):
    # Use fillna(method='ffill') first for forward fill
    df = df.ffill()
    # Then fill remaining NaNs (e.g., at the beginning) with the median
    # Specify numeric_only=True to avoid errors on non-numeric columns
    df.fillna(df.median(numeric_only=True), inplace=True)
    return df

def create_delta_columns(df):
    open_chem = ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']
    final_chem = [f"F-{el}" for el in ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']]
    # Ensure columns exist before attempting calculation
    for open_col, final_col in zip(open_chem, final_chem):
        if open_col in df.columns and final_col in df.columns:
            delta_col = f"Delta_{open_col.replace('%', '')}"
            # Ensure columns are numeric before subtraction
            # Use errors='coerce' to turn non-numeric values into NaN, then fill NaN
            df[open_col] = pd.to_numeric(df[open_col], errors='coerce')
            df[final_col] = pd.to_numeric(df[final_col], errors='coerce')
            df[delta_col] = df[final_col] - df[open_col]
            # Fill NaNs that might result from coercion or original NaNs
            df[delta_col].fillna(df[delta_col].median(), inplace=True) # Fill with median of the delta column
    return df

def clip_outliers(df):
    # Select only numeric columns for clipping
    numeric_cols = df.select_dtypes(include=np.number).columns
    for col in numeric_cols:
        # Calculate quantiles and IQR, ignoring NaNs
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        # Clip values in the column
        df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
    return df

def preprocess_pipeline(filepath):
    df = load_data(filepath)
    summary_df = load_summary(filepath) # summary_df is loaded here but not used in subsequent steps within this function
    df = handle_missing(df)
    df = create_delta_columns(df)
    df = clip_outliers(df)
    # Note: This pipeline returns the processed df and the summary_df separately
    return df, summary_df
# --- End Re-defining Preprocessing Functions ---


# 1. Preprocessing and Scaling
filepath = "FE Alloying.xlsx"
df, summary_df = preprocess_pipeline(filepath)

# Define alloy and process features
alloy_cols = [
    "CSP-SiMn", "Mn HC", "Mn MC", "Mn LC", "Mn Metal", "FeSi", "Ladle Cov",
    "FeMo Metal", "FeV", "FeNb lumps", "FeTi lumps", "FeTi Wire", "FeB", "FeAl",
    "Cal Carb", "Al bar", "Al  wire", "FeP", "Sul Stick", "Al mix", "CaSi wire",
    "Cal Wire", "CaFeAl Wire", "S Wire", "Ni Plate", "FeCr LC", "FeCr HC",
    "Al Shot", "Lead Wire", "Mo Metal", "Syn Slag"
]
# Add Delta columns to the potential list of process/chemistry features
process_chem_cols_base = [
    'Lift Temp', 'Liquidus temp (° C)', 'Arching Time-mm',
    'LRF Holding Time-mm', 'LRF Lime',
    'C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%'
]
delta_cols = [f"Delta_{el.replace('%','')}" for el in process_chem_cols_base if f"Delta_{el.replace('%','')}" in df.columns]

# The features used for the model should include original process/open chem AND delta columns
features = [col for col in alloy_cols + process_chem_cols_base + delta_cols if col in df.columns]

target_cols = [f"F-{el}" for el in ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']]
# Filter target_cols to only include those present in the dataframe
target = [col for col in target_cols if col in df.columns]


# Select data using the filtered features and target lists
X = df[features]
y = df[target]

# Handle datetime.time columns and other non-numeric types by coercing to numeric
# This step needs to be applied consistently to X and any future input dataframes
for col in X.select_dtypes(include=['object']).columns:
    try:
        # Ensure this conversion method matches the one used later in evaluate_alloy_additions
        # Convert to string first to handle mixed types or datetime.time
        X[col] = pd.to_numeric(X[col].astype(str), errors='coerce').fillna(0)
    except Exception as e:
        print(f"Could not convert column {col} to numeric in X: {e}")


# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
# Fit scaler ONLY on the training features (X_train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Scale target variables separately
target_scaler = StandardScaler()
y_train_scaled = target_scaler.fit_transform(y_train) # Fit the target scaler on original y_train
y_test_scaled = target_scaler.transform(y_test)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

# Create DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 2. PyTorch TabTransformer Model (adapted for numerical features)
class NumericalTabTransformer(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=128, num_layers=2, num_heads=2):
        super(NumericalTabTransformer, self).__init__()

        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()

        # Simulate Transformer layers with Dense layers and potential feature mixing
        # Use LayerNorm before residual connection as is common in Transformers
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim), # Another linear layer for transformation
                nn.LayerNorm(hidden_dim) # Add Layer Normalization
            ) for _ in range(num_layers)
        ])

        self.output_layer = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.relu(self.input_layer(x))
        # Pass through transformer-like layers
        for layer in self.layers:
            # Apply layer and add residual connection
            x = x + layer(x)
        x = self.output_layer(x)
        return x

input_dim = X_train_scaled.shape[1]
output_dim = y_train_scaled.shape[1] # Output dimension matches the scaled target variables
model = NumericalTabTransformer(input_dim, output_dim)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. Model Training
num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}") # Suppress for cleaner output


# Save the trained PyTorch model
os.makedirs("models", exist_ok=True) # Ensure models directory exists
torch.save(model.state_dict(), "models/numerical_tab_transformer.pth")
print("Numerical TabTransformer model saved.")

# Model Evaluation (RMSE, R²)
model.eval()
with torch.no_grad():
    predictions = []
    actuals = []
    for inputs, targets in test_loader:
        outputs = model(inputs)
        predictions.append(outputs.numpy())
        actuals.append(targets.numpy())

predictions = np.concatenate(predictions)
actuals = np.concatenate(actuals)

# Inverse transform the scaled predictions and actuals using the target scaler
predictions_inv_scaled = target_scaler.inverse_transform(predictions)
actuals_inv_scaled = target_scaler.inverse_transform(actuals)

# Calculate RMSE
rmse = np.sqrt(np.mean((predictions_inv_scaled - actuals_inv_scaled)**2))

# Calculate R²
from sklearn.metrics import r2_score
r2 = r2_score(actuals_inv_scaled, predictions_inv_scaled)

print(f"Test RMSE (Inverse Scaled): {rmse:.4f}")
print(f"Test R² (Inverse Scaled): {r2:.4f}")

# 4. Genetic Algorithm Optimization using DEAP

# Load the trained PyTorch model for optimization
loaded_model = NumericalTabTransformer(input_dim, output_dim)
loaded_model.load_state_dict(torch.load("models/numerical_tab_transformer.pth"))
loaded_model.eval() # Set to evaluation mode

# Get base inputs from median of successful heats (reusing previous logic)
# This should use the SAME list of feature columns ('features') that the model was trained on
max_summary_row = summary_df.iloc[2].fillna(0)
aim_summary_row = summary_df.iloc[3].fillna(0)
min_summary_row = summary_df.iloc[1].fillna(0)

target_keys_summary = [k.strip() for k in max_summary_row.index if k.strip().endswith("%")]
target_vector_summary = max_summary_row[target_keys_summary].dropna().astype(float)
# Ensure 'F-' columns exist in the dataframe before using them for Success_Score calculation
present_f_cols_summary = [f"F-{k}" for k in target_keys_summary if f"F-{k}" in df.columns]

# Calculate Success_Score only if relevant 'F-' columns are present
if present_f_cols_summary:
     df['Success_Score'] = -((df[present_f_cols_summary] - target_vector_summary.loc[[k for k in target_keys_summary if f"F-{k}" in df.columns]].values) ** 2).sum(axis=1) ** 0.3
     df_successful = df[df['Success_Score'] >= df['Success_Score'].quantile(0.5)]
else:
     print("Warning: No relevant 'F-%' columns found for success score calculation. Using entire df for base inputs/bounds.")
     df_successful = df.copy() # Fallback to using entire df if no F-% columns


# Select base inputs from median of successful heats using the 'features' list
# This ensures base_inputs_opt starts with a dictionary containing keys from 'features'
# Calculate median only for columns present in df_successful and 'features'
cols_for_base_inputs = [col for col in features if col in df_successful.columns]
base_inputs_opt_df = df_successful[cols_for_base_inputs].median(numeric_only=True)
base_inputs_opt = base_inputs_opt_df.to_dict()


# Extract target chemistry for optimization (using max_chem or aim_chem)
# Ensure the order of target_chem_opt matches the order of target_cols for RMSE calculation later
max_chem = {
    f"F-{k.strip()}": float(v) for k, v in max_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v) and f"F-{k.strip()}" in target
}
aim_chem = {
    f"F-{k.strip()}": float(v) for k, v in aim_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v) and f"F-{k.strip()}" in target
}
min_chem = {
    f"F-{k.strip()}": float(v) for k, v in min_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v) and f"F-{k.strip()}" in target
}


# Use max_chem for optimization target values, ensuring they are in the order of target_cols
# This dictionary will be used to create the target vector for RMSE calculation
target_chem_opt_dict = {col: max_chem.get(col, 0) for col in target} # Use 'target' which is the filtered target_cols


# Bounds for alloy elements for optimization
# Only use bounds for the alloy_cols that are actually included in the model features
alloy_features_in_model = [col for col in alloy_cols if col in features]

# Calculate bounds based on df_successful, but enforce a minimum of 0 for alloy additions
bounds = [(max(0, df_successful[el].min()), df_successful[el].max()) for el in alloy_features_in_model]

# Adjust bounds to ensure min < max, add a small epsilon if min == max and min is 0
bounds = [(lower, upper) if lower < upper else (lower, lower + 1e-6 if lower == 0 else lower + abs(lower)*1e-6) for lower, upper in bounds]

# Extract bounds into low and up lists for the custom mutation operator
low_bounds = [b[0] for b in bounds]
up_bounds = [b[1] for b in bounds]

# DEAP setup (re-create if needed in this cell)
try:
    # Attempt to delete existing definitions if running cell multiple times
    del creator.FitnessMin
    del creator.Individual
except AttributeError:
    pass # Ignore if they don't exist

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
# Register the individual initializer to use the actual bounds directly
toolbox.register("individual", tools.initIterate, creator.Individual,
                 lambda: [random.uniform(low, high) for low, high in bounds]) # Initialize within actual bounds
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("mate", tools.cxBlend, alpha=0.5)

# Define the custom mutation operator that respects bounds
def mutGaussianBounded(individual, mu, sigma, indpb, low, up):
    size = len(individual)
    for i in range(size):
        if random.random() < indpb:
            # Apply Gaussian mutation
            mutated_value = individual[i] + random.gauss(mu, sigma)
            # Clip to bounds
            individual[i] = np.clip(mutated_value, low[i], up[i])
    return individual,

# Register the bounded mutation operator
toolbox.register("mutate", mutGaussianBounded, mu=0, sigma=1, indpb=0.2, low=low_bounds, up=up_bounds)
toolbox.register("select", tools.selTournament, tournsize=3)


# Register the evaluation function using the trained PyTorch model
def evaluate_alloy_additions(individual):
    # The individual already contains alloy values for the features in alloy_features_in_model
    alloy_values = individual # Individual is a list of floats corresponding to alloy_features_in_model

    # Create a dictionary with the alloy values from the individual
    alloy_dict = dict(zip(alloy_features_in_model, alloy_values))

    # Combine base inputs with the current individual's alloy additions
    # Start with a copy of base_inputs_opt (which contains median of 'features')
    input_data_for_scaling = base_inputs_opt.copy()
    # Update with the alloy values from the current individual's alloy additions
    # This overwrites the median alloy values from base_inputs_opt with the current individual's values
    input_data_for_scaling.update(alloy_dict)

    # Create a DataFrame ensuring all and only the 'features' columns are present
    # Use the exact 'features' list which was used to create X_train and fit the scaler
    input_df = pd.DataFrame([input_data_for_scaling])

    # --- DEBUG PRINT STATEMENTS ---
    # print("\n--- Debugging scaler input ---")
    # print("Expected features (from X_train.columns or 'features'):", features)
    # print("Input DataFrame columns:", input_df.columns.tolist())
    # print("Are columns identical and in same order?", features == input_df.columns.tolist())
    # print("Missing features in input_df:", set(features) - set(input_df.columns))
    # print("Extra features in input_df:", set(input_df.columns) - set(features))
    # print("--------------------------")
    # --- END DEBUG PRINT STATEMENTS ---

    # ***FIX***: Ensure the DataFrame columns match the exact features used for training
    # Reindex to the order of 'features' and fill any potentially missing with 0
    # This step is crucial for the scaler
    input_df = input_df.reindex(columns=features, fill_value=0)

    # Handle datetime.time columns and other non-numeric types by coercing to numeric
    # This step must exactly mirror how X_train was handled before scaling
    for col in input_df.select_dtypes(include=['object']).columns:
        try:
            # Use the same conversion method as applied to X before splitting
            input_df[col] = pd.to_numeric(input_df[col].astype(str), errors='coerce').fillna(0)
        except Exception as e:
             # Print error but continue, coercing to numeric should handle most issues
             print(f"Warning: Could not convert column {col} to numeric in evaluate_alloy_additions: {e}")
             input_df[col] = pd.to_numeric(input_df[col], errors='coerce').fillna(0)


    # Scale the input using the same scaler used during training
    # The scaler was fitted on X_train, which contains the 'features' columns in order
    input_scaled = scaler.transform(input_df)
    input_tensor = torch.tensor(input_scaled, dtype=torch.float32)

    # Make prediction using the loaded PyTorch model
    with torch.no_grad(): # No gradient calculation needed for evaluation
        predicted_scaled = loaded_model(input_tensor).numpy()[0]

    # Inverse transform the scaled prediction to original scale
    # Use the target_scaler which was fitted specifically on the target variables (y_train)
    # predicted_scaled is a 1D array corresponding to the scaled target columns
    predicted_chem = target_scaler.inverse_transform(predicted_scaled.reshape(1, -1))[0]


    # Calculate the objective (RMSE against target chemistry)
    # Use the target_chem_opt_dict to get target values in the correct order (matching 'target' list)
    target_values = np.array([target_chem_opt_dict.get(col, 0) for col in target])

    # Ensure prediction and target have the same number of elements for comparison
    # The number of elements in predicted_chem should match the number of 'target' columns
    num_elements_to_compare = len(target) # Use the length of the filtered target columns
    predicted_chem_sliced = predicted_chem[:num_elements_to_compare]
    target_values_sliced = target_values[:num_elements_to_compare]

    # Calculate RMSE
    # Add a small epsilon to avoid log(0) if using other metrics, but RMSE is fine with 0
    rmse = np.sqrt(np.mean((predicted_chem_sliced - target_values_sliced)**2))

    return rmse, # Comma for DEAP tuple

# Register the evaluation function with DEAP
toolbox.register("evaluate", evaluate_alloy_additions)

# Run Genetic Algorithm
pop = toolbox.population(n=50)
hof = tools.HallOfFame(1)
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("min", np.min)

print("Starting Genetic Algorithm Optimization...")
# Run the GA
# Use HallOfFame as a list so we can access the best individual after the run# Run the GA
logbook = algorithms.eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=100, stats=stats, halloffame=hof, verbose=False) # Corrected keyword argument to halloffame

print("Genetic Algorithm Optimization Finished.")

# Best solution from GA is in hof[0]
if hof:
    best_individual = hof[0]
else:
    # Handle case where hall of fame might be empty (e.g., ngen=0)
    print("Warning: Hall of Fame is empty. Cannot retrieve best individual.")
    best_individual = None
    optimized_alloy_additions = {}


if best_individual is not None:
    # Map the optimized values back to the alloy column names
    # best_individual is a list of optimized values corresponding to alloy_features_in_model
    optimized_alloy_additions_raw = dict(zip(alloy_features_in_model, best_individual))

    # ***FIX: Ensure all final alloy additions are non-negative by clipping at 0***
    optimized_alloy_additions = {key: max(0.0, value) for key, value in optimized_alloy_additions_raw.items()}


    print("\nOptimized Alloy Additions (TabTransformer + GA):")
    # Display as DataFrame for better readability, including all original alloy_cols
    display_optimized_alloys = {col: optimized_alloy_additions.get(col, 0.0) for col in alloy_cols}
    # Ensure the values are displayed with appropriate precision and handle the index name
    display(pd.DataFrame(display_optimized_alloys, index=["Recommended (kg)"]).T.round(4))


    # Predict the final chemistry for the optimized alloys using the trained model
    # Recreate the input DataFrame using the optimized alloys and base inputs
    optimized_input_dict = base_inputs_opt.copy() # Start with base inputs (median of features)
    # Update with the *clipped* optimized alloy values
    optimized_input_dict.update(optimized_alloy_additions)

    # Create DataFrame with the exact 'features' columns
    optimized_input_df = pd.DataFrame([optimized_input_dict]).reindex(columns=features, fill_value=0)

    # Handle non-numeric types consistently
    for col in optimized_input_df.select_dtypes(include=['object']).columns:
        try:
            optimized_input_df[col] = pd.to_numeric(optimized_input_df[col].astype(str), errors='coerce').fillna(0)
        except Exception as e:
             print(f"Warning: Could not convert column {col} to numeric for final prediction: {e}")
             optimized_input_df[col] = pd.to_numeric(optimized_input_df[col], errors='coerce').fillna(0)


    # Scale the input using the feature scaler
    optimized_input_scaled = scaler.transform(optimized_input_df)
    optimized_input_tensor = torch.tensor(optimized_input_scaled, dtype=torch.float32)

    with torch.no_grad():
        predicted_scaled_opt = loaded_model(optimized_input_tensor).numpy()[0]

    # Inverse transform using the target scaler
    predicted_chem_opt = target_scaler.inverse_transform(predicted_scaled_opt.reshape(1, -1))[0]


    print("\nPredicted Final Chemistry vs Target (using optimized alloys):")
    # Align predicted chemistry with target_cols for comparison
    # Create a dictionary from the predicted_chem_opt array using the target column names
    predicted_chem_dict = dict(zip(target, predicted_chem_opt[:len(target)])) # Use the filtered 'target' list

    # Create comparison DataFrame
    chem_comparison_df = pd.DataFrame({
        "Target (Max)": [max_chem.get(col, 0) for col in target], # Use max_chem and filtered 'target' list
        "Aim": [aim_chem.get(col, 0) for col in target],       # Use aim_chem and filtered 'target' list
        "Min": [min_chem.get(col, 0) for col in target],       # Use min_chem and filtered 'target' list
        "Predicted": [predicted_chem_dict.get(col, 0) for col in target] # Use predicted_chem_dict and filtered 'target' list
    }, index=target) # Use the filtered 'target' list as index

    display(chem_comparison_df.round(5))
else:
    print("Optimization did not produce a valid best individual.")

<ipython-input-127-1c7286f1bc92>:41: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.ffill()
<ipython-input-127-1c7286f1bc92>:60: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[delta_col].fillna(df[delta_col].median(), inplace=True) # Fill with median of the delta column
<ipython-input-127-1c7286f1bc92>

Numerical TabTransformer model saved.
Test RMSE (Inverse Scaled): 0.0014
Test R² (Inverse Scaled): -326896025600.0000
Starting Genetic Algorithm Optimization...
Genetic Algorithm Optimization Finished.

Optimized Alloy Additions (TabTransformer + GA):


,Recommended (kg)
CSP-SiMn,0.0000
Mn HC,0.0000
Mn MC,0.0000
Mn LC,0.0000
Mn Metal,0.0000
FeSi,0.0000
Ladle Cov,0.0000
FeMo Metal,0.0000
FeV,0.0000
FeNb lumps,0.0000



Predicted Final Chemistry vs Target (using optimized alloys):


,Target (Max),Aim,Min,Predicted
F-C%,0.060,0.050,0.02,0.04179
F-Mn%,0.150,0.120,0.10,0.13980
F-S%,0.010,0.000,0.00,0.00642
F-P%,0.030,0.015,0.00,0.01390
F-Si%,0.050,0.040,0.00,0.02581
F-Cr%,0.050,0.000,0.00,0.01338
F-Ni%,0.050,0.000,0.00,0.00324
F-Mo%,0.005,0.000,0.00,0.00040
F-V%,0.003,0.000,0.00,-0.00001
F-Ti%,0.003,0.000,0.00,0.00097


In [ ]:
!pip install transformers

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
import tensorflow as tf
import os  # Import the os module
from IPython import get_ipython
from IPython.display import display
import datetime # Import the datetime module

# 1. Load and preprocess your data
# Assuming 'filepath' is defined and points to your Excel file
df, summary_df = preprocess_pipeline(filepath)

# Define alloy and process features
alloy_cols = [
    "CSP-SiMn", "Mn HC", "Mn MC", "Mn LC", "Mn Metal", "FeSi", "Ladle Cov",
    "FeMo Metal", "FeV", "FeNb lumps", "FeTi lumps", "FeTi Wire", "FeB", "FeAl",
    "Cal Carb", "Al bar", "Al  wire", "FeP", "Sul Stick", "Al mix", "CaSi wire",
    "Cal Wire", "CaFeAl Wire", "S Wire", "Ni Plate", "FeCr LC", "FeCr HC",
    "Al Shot", "Lead Wire", "Mo Metal", "Syn Slag"
]
process_chem_cols = [
    'Lift Temp', 'Liquidus temp (° C)', 'Arching Time-mm',
    'LRF Holding Time-mm', 'LRF Lime',
    'C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%'
]
target_cols = [f"F-{el}" for el in ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']]

# Select features and target
features = alloy_cols + process_chem_cols  # Use existing alloy and process features
target = target_cols  # Use final chemistry as target

X = df[features]
y = df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# *** Convert datetime.time columns to numerical representation ***
for col in X_train.select_dtypes(include=['object']).columns:
    if X_train[col].apply(lambda x: isinstance(x, (datetime.time))).any():
        X_train[col] = pd.to_numeric(X_train[col].astype(str), errors='coerce').fillna(0)
        X_test[col] = pd.to_numeric(X_test[col].astype(str), errors='coerce').fillna(0)

# 2. Build the Transformer model (adapted for numerical features)
# For numerical features, we'll skip the tokenizer and use a simple input layer
# We'll also use a MultiOutputRegressor for multiple target properties

def build_transformer_model(input_shape):
    input_layer = tf.keras.layers.Input(shape=input_shape, dtype=tf.float32, name="numerical_features")
    # You might consider adding layers like Dense or normalization here if
    # needed
    output_layer = tf.keras.layers.Dense(len(target_cols), activation='linear')(input_layer) # Output layer for multiple targets
    model = tf.keras.Model(inputs=input_layer, outputs=output_layer)
    return model

# 3. Train the model
input_shape = (len(features),)
model = build_transformer_model(input_shape)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5), loss='mse', metrics=['mae'])
model.fit(X_train, y_train, epochs=3, batch_size=32, validation_split=0.1)

# 4. Evaluate the model
loss, mae = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {loss:.4f}, Test MAE: {mae:.4f}")

# 5. Save and use the model for predictions
model_dir = 'models'  # Define a directory for saving the model

# Create the directory if it doesn't exist
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

# **Change:** Provide a filename with the .keras extension
model_path = os.path.join(model_dir, 'my_transformer_model.keras')
model.save(model_path)  # Save the model to the specified path with filename


# # To load and use the model for prediction:
# loaded_model = tf.keras.models.load_model(model_dir)  # Load the saved model

# # Prepare new data for prediction (similar to X_test)
# new_data = ...  # Replace with your new data

# # Make predictions
# predictions = loaded_model.predict(new_data)

In [125]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from deap import base, creator, tools, algorithms
import random
import joblib # Make sure joblib is imported
import os
import datetime
from IPython.display import display
from sklearn.metrics import mean_squared_error # Ensure this is imported if needed

# --- Re-defining Preprocessing Functions (Ensure these match your other blocks if needed) ---
# Re-define the preprocessing functions if they are not guaranteed to be in scope
# from src.preprocessing import load_data, load_summary, preprocess_data # If using external file
# Otherwise, include the function definitions directly here if they are modified
def load_data(filepath):
    # Use os.path.basename to get the filename for checking the extension
    df = pd.read_excel(filepath, sheet_name="Heats") if os.path.basename(filepath).endswith('.xlsx') else pd.read_csv(filepath)
    df.columns = df.columns.str.strip()
    if df.iloc[0].isnull().sum() < 5:
        df.columns = df.iloc[0]
        df = df[1:]
        df.columns = df.columns.str.strip()
    # Avoid inplace=True and chained assignment warnings by chaining calls and assigning back
    # Also add .copy() after dropping rows/columns to avoid SettingWithCopyWarning later
    df = df.dropna(axis=1, how='all').dropna(axis=0, how='all').copy()
    return df

def load_summary(filepath):
    # Use os.path.basename to extract filename from filepath
    if os.path.basename(filepath).endswith('.xlsx'): # Use os.path.basename to extract filename
        summary_df = pd.read_excel(filepath, sheet_name="Summary")
        summary_df.columns = summary_df.columns.str.strip()
        # Avoid inplace=True and add .copy()
        summary_df = summary_df.dropna(how='all').copy()
        return summary_df
    return None

def handle_missing(df):
    # Avoid inplace=True and chained assignment warnings
    # ***FIX: Replace fillna(method='ffill') with ffill()***
    df = df.ffill() # Use ffill() directly
    # Then fill remaining NaNs with the median, specifying numeric_only=True
    # Use the result of fillna directly and assign back to df
    # Add .infer_objects(copy=False) to address downcasting warning
    df = df.fillna(df.median(numeric_only=True)).infer_objects(copy=False)
    return df # Return the modified DataFrame

def create_delta_columns(df):
    # Avoid modifying input df directly with inplace=True and chained assignment warnings
    df_copy = df.copy() # Work on a copy to prevent modifying the original df unexpectedly
    open_chem = ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']
    final_chem = [f"F-{el}" for el in ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']]
    # Ensure columns exist before attempting calculation
    for open_col, final_col in zip(open_chem, final_chem):
        if open_col in df_copy.columns and final_col in df_copy.columns:
            delta_col = f"Delta_{open_col.replace('%', '')}"
            # Ensure columns are numeric before subtraction
            # Use errors='coerce' to turn non-numeric values into NaN
            df_copy[open_col] = pd.to_numeric(df_copy[open_col], errors='coerce')
            df_copy[final_col] = pd.to_numeric(df_copy[final_col], errors='coerce')
            df_copy[delta_col] = df_copy[final_col] - df_copy[open_col]
            # Fill NaNs that might result from coercion or original NaNs
            # Calculate median of the delta column and use fillna on the column directly
            median_delta = df_copy[delta_col].median()
            # Avoid inplace=True
            df_copy[delta_col] = df_copy[delta_col].fillna(median_delta)

    return df_copy # Return the modified copy of the DataFrame

def clip_outliers(df):
    # Avoid modifying input df directly with inplace=True warnings
    df_copy = df.copy() # Work on a copy
    # Select only numeric columns for clipping
    numeric_cols = df_copy.select_dtypes(include=np.number).columns
    for col in numeric_cols:
        # Calculate quantiles and IQR, ignoring NaNs
        q1 = df_copy[col].quantile(0.25)
        q3 = df_copy[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        # Clip values in the column and assign back
        # Avoid inplace=True
        df_copy[col] = df_copy[col].clip(lower=lower_bound, upper=upper_bound)
    return df_copy # Return the modified copy

def preprocess_pipeline(filepath):
    df = load_data(filepath)
    summary_df = load_summary(filepath)
    # Chain the function calls and reassign the result
    # Each function should return a new or modified DataFrame
    df = handle_missing(df)
    df = create_delta_columns(df)
    df = clip_outliers(df)
    # Note: This pipeline returns the processed df and the summary_df separately
    return df, summary_df

# Assuming preprocess_data is also used elsewhere and might need similar updates
def preprocess_data(df):
    # Avoid modifying input df directly
    df = df.copy()
    # Convert datetime or object to numeric where needed
    for col in df.columns:
        if df[col].dtype == 'O':  # If the column is of object (string) type
            try:
                # Specify the format of your date/time column here (if known)
                # Use errors='coerce' and fillna to handle non-datetime strings gracefully
                # Assign the result back to the column
                df[col] = pd.to_datetime(df[col], errors='coerce').dt.total_seconds().fillna(0)
            except Exception: # Catch other potential errors during conversion
                 try:
                      # Handle cases where the column is not a date/time but might be numeric strings
                      # Assign the result back to the column
                      df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
                 except Exception as e:
                      print(f"Could not convert column {col} to numeric in preprocess_data: {e}")
                      # Assign a default value or leave as NaN if conversion fails completely
                      df[col] = np.nan # Assign NaN then the fillna(0) below will handle it


    # Fill remaining NaNs with 0 (as done in your original preprocess_data)
    # Avoid inplace=True
    df = df.fillna(0)

    # Outlier clipping
    # Avoid modifying input df directly
    # Select only numeric columns for clipping
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        # Clip values in the column and assign back
        # Avoid inplace=True
        df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

    return df # Return the modified DataFrame

# --- End Corrected Preprocessing Functions ---


# --- Section where X and y are created (Modify this) ---

# 1. Preprocessing and Scaling
filepath = "FE Alloying.xlsx"
df, summary_df = preprocess_pipeline(filepath)

# Define alloy and process features
# ... (alloy_cols, process_chem_cols_base, delta_cols, features, target definitions remain the same)
alloy_cols = [
    "CSP-SiMn", "Mn HC", "Mn MC", "Mn LC", "Mn Metal", "FeSi", "Ladle Cov",
    "FeMo Metal", "FeV", "FeNb lumps", "FeTi lumps", "FeTi Wire", "FeB", "FeAl",
    "Cal Carb", "Al bar", "Al  wire", "FeP", "Sul Stick", "Al mix", "CaSi wire",
    "Cal Wire", "CaFeAl Wire", "S Wire", "Ni Plate", "FeCr LC", "FeCr HC",
    "Al Shot", "Lead Wire", "Mo Metal", "Syn Slag"
]
process_chem_cols_base = [
    'Lift Temp', 'Liquidus temp (° C)', 'Arching Time-mm',
    'LRF Holding Time-mm', 'LRF Lime',
    'C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%'
]
delta_cols = [f"Delta_{el.replace('%','')}" for el in process_chem_cols_base]
features = [col for col in alloy_cols + process_chem_cols_base + delta_cols if col in df.columns]

target_cols = [f"F-{el}" for el in ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']]
target = [col for col in target_cols if col in df.columns]


# Select data using the filtered features and target lists
# ***FIX: Explicitly create a copy of X to avoid SettingWithCopyWarning***
X = df[features].copy() # Use .copy() here
y = df[target].copy() # Use .copy() here as well for consistency


# Handle datetime.time columns and other non-numeric types by coercing to numeric
# This step needs to be applied consistently to X and any future input dataframes
for col in X.select_dtypes(include=['object']).columns:
    try:
        # Ensure this conversion method matches the one used later in evaluate_alloy_additions
        # Convert to string first to handle mixed types or datetime.time
        # Assign the result back to the column in the copied DataFrame X
        X[col] = pd.to_numeric(X[col].astype(str), errors='coerce').fillna(0)
    except Exception as e:
        print(f"Could not convert column {col} to numeric in X: {e}")


# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
# Fit scaler ONLY on the training features (X_train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Scale target variables separately
target_scaler = StandardScaler()
y_train_scaled = target_scaler.fit_transform(y_train) # Fit the target scaler on original y_train
y_test_scaled = target_scaler.transform(y_test)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

# Create DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 2. PyTorch TabTransformer Model (adapted for numerical features)
class NumericalTabTransformer(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=128, num_layers=2, num_heads=2):
        super(NumericalTabTransformer, self).__init__()

        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()

        # Simulate Transformer layers with Dense layers and potential feature mixing
        # Use LayerNorm before residual connection as is common in Transformers
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim), # Another linear layer for transformation
                nn.LayerNorm(hidden_dim) # Add Layer Normalization
            ) for _ in range(num_layers)
        ])

        self.output_layer = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.relu(self.input_layer(x))
        # Pass through transformer-like layers
        for layer in self.layers:
            # Apply layer and add residual connection
            x = x + layer(x)
        x = self.output_layer(x)
        return x

input_dim = X_train_scaled.shape[1]
output_dim = y_train_scaled.shape[1] # Output dimension matches the scaled target variables
model = NumericalTabTransformer(input_dim, output_dim)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. Model Training
num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}") # Suppress for cleaner output


# Save the trained PyTorch model and the scalers
os.makedirs("models", exist_ok=True) # Ensure models directory exists

# Define the paths where you want to save the model and scalers
model_path = "models/numerical_tab_transformer.pth"
feature_scaler_path = "models/feature_scaler.pkl"
target_scaler_path = "models/target_scaler.pkl"


torch.save(model.state_dict(), model_path)
print(f"Numerical TabTransformer model saved to: {model_path}")

# ***ADD THESE LINES TO SAVE SCALERS***
joblib.dump(scaler, feature_scaler_path) # Save the feature scaler
joblib.dump(target_scaler, target_scaler_path) # Save the target scaler
print(f"Feature scaler saved to: {feature_scaler_path}")
print(f"Target scaler saved to: {target_scaler_path}")
# ************************************


# Model Evaluation (RMSE, R²)
model.eval()
with torch.no_grad():
    predictions = []
    actuals = []
    for inputs, targets in test_loader:
        outputs = model(inputs)
        predictions.append(outputs.numpy())
        actuals.append(targets.numpy())

predictions = np.concatenate(predictions)
actuals = np.concatenate(actuals)

# Inverse transform the scaled predictions and actuals using the target scaler
predictions_inv_scaled = target_scaler.inverse_transform(predictions)
actuals_inv_scaled = target_scaler.inverse_transform(actuals)

# Calculate RMSE
rmse = np.sqrt(np.mean((predictions_inv_scaled - actuals_inv_scaled)**2))

# Calculate R²
from sklearn.metrics import r2_score
r2 = r2_score(actuals_inv_scaled, predictions_inv_scaled)

print(f"Test RMSE (Inverse Scaled): {rmse:.4f}")
print(f"Test R² (Inverse Scaled): {r2:.4f}")

# 4. Genetic Algorithm Optimization using DEAP

# Load the trained PyTorch model for optimization
loaded_model = NumericalTabTransformer(input_dim, output_dim)
loaded_model.load_state_dict(torch.load(model_path)) # Load from the defined path
loaded_model.eval() # Set to evaluation mode

# Load the scalers for use in the evaluation function
loaded_feature_scaler = joblib.load(feature_scaler_path) # Load feature scaler
loaded_target_scaler = joblib.load(target_scaler_path) # Load target scaler


# Get base inputs from median of successful heats (reusing previous logic)
# This should use the SAME list of feature columns ('features') that the model was trained on
max_summary_row = summary_df.iloc[2].fillna(0)
aim_summary_row = summary_df.iloc[3].fillna(0)
min_summary_row = summary_df.iloc[1].fillna(0)

target_keys_summary = [k.strip() for k in max_summary_row.index if k.strip().endswith("%")]
target_vector_summary = max_summary_row[target_keys_summary].dropna().astype(float)
# Ensure 'F-' columns exist in the dataframe before using them for Success_Score calculation
present_f_cols_summary = [f"F-{k}" for k in target_keys_summary if f"F-{k}" in df.columns]

# Calculate Success_Score only if relevant 'F-' columns are present
if present_f_cols_summary:
     df['Success_Score'] = -((df[present_f_cols_summary] - target_vector_summary.loc[[k for k in target_keys_summary if f"F-{k}" in df.columns]].values) ** 2).sum(axis=1) ** 0.3
     df_successful = df[df['Success_Score'] >= df['Success_Score'].quantile(0.5)].copy() # Use .copy()
else:
     print("Warning: No relevant 'F-%' columns found for success score calculation. Using entire df for base inputs/bounds.")
     df_successful = df.copy() # Fallback to using entire df if no F-% columns


# Select base inputs from median of successful heats using the 'features' list
# This ensures base_inputs_opt starts with a dictionary containing keys from 'features'
# Calculate median only for columns present in df_successful and 'features'
cols_for_base_inputs = [col for col in features if col in df_successful.columns]
base_inputs_opt_df = df_successful[cols_for_base_inputs].median(numeric_only=True)
base_inputs_opt = base_inputs_opt_df.to_dict()


# Extract target chemistry for optimization (using max_chem or aim_chem)
# Ensure the order of target_chem_opt matches the order of target_cols for RMSE calculation later
max_chem = {
    f"F-{k.strip()}": float(v) for k, v in max_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v) and f"F-{k.strip()}" in target # Filter to only include targets in the model output
}
aim_chem = {
    f"F-{k.strip()}": float(v) for k, v in aim_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v) and f"F-{k.strip()}" in target
}
min_chem = {
    f"F-{k.strip()}": float(v) for k, v in min_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v) and f"F-{k.strip()}" in target
}


# ***CHANGE HERE: Use aim_chem for the optimization target***
target_chem_opt_dict = {col: aim_chem.get(col, 0) for col in target} # Use 'aim_chem' as the target dictionary for optimization


# Bounds for alloy elements for optimization
# Only use bounds for the alloy_cols that are actually included in the model features
alloy_features_in_model = [col for col in alloy_cols if col in features]

# Calculate bounds based on df_successful, but enforce a minimum of 0 for alloy additions
# Ensure the columns exist in df_successful before attempting to get min/max
bounds = [(max(0, df_successful[el].min()) if el in df_successful.columns else 0.0,
           df_successful[el].max() if el in df_successful.columns else 100.0) # Default max if column missing
          for el in alloy_features_in_model]

# Adjust bounds to ensure min < max, add a small epsilon if min == max and min is 0
bounds = [(lower, upper) if lower < upper else (lower, lower + 1e-6 if lower == 0 else lower + abs(lower)*1e-6) for lower, upper in bounds]

# Extract bounds into low and up lists for the custom mutation operator
low_bounds = [b[0] for b in bounds]
up_bounds = [b[1] for b in bounds]

# DEAP setup (re-create if needed in this cell)
try:
    # Attempt to delete existing definitions if running cell multiple times
    del creator.FitnessMin
    del creator.Individual
except AttributeError:
    pass # Ignore if they don't exist

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
# Register the individual initializer to use the actual bounds directly
toolbox.register("individual", tools.initIterate, creator.Individual,
                 lambda: [random.uniform(low, high) for low, high in bounds]) # Initialize within actual bounds
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("mate", tools.cxBlend, alpha=0.5)

# Define the custom mutation operator that respects bounds
def mutGaussianBounded(individual, mu, sigma, indpb, low, up):
    size = len(individual)
    for i in range(size):
        if random.random() < indpb:
            # Apply Gaussian mutation
            mutated_value = individual[i] + random.gauss(mu, sigma)
            # Clip to bounds
            individual[i] = np.clip(mutated_value, low[i], up[i])
    return individual,

# Register the bounded mutation operator
toolbox.register("mutate", mutGaussianBounded, mu=0, sigma=1, indpb=0.2, low=low_bounds, up=up_bounds)
toolbox.register("select", tools.selTournament, tournsize=3)


# Register the evaluation function using the trained PyTorch model
def evaluate_alloy_additions(individual):
    # The individual already contains alloy values for the features in alloy_features_in_model
    alloy_values = individual # Individual is a list of floats corresponding to alloy_features_in_model

    # Create a dictionary with the alloy values from the individual
    alloy_dict = dict(zip(alloy_features_in_model, alloy_values))

    # Combine base inputs with the current individual's alloy additions
    # Start with a copy of base_inputs_opt (which contains median of 'features')
    input_data_for_scaling = base_inputs_opt.copy()
    # Update with the alloy values from the current individual's alloy additions
    # This overwrites the median alloy values from base_inputs_opt with the current individual's values
    input_data_for_scaling.update(alloy_dict)

    # Create a DataFrame ensuring all and only the 'features' columns are present
    # Use the exact 'features' list which was used to create X_train and fit the scaler
    input_df = pd.DataFrame([input_data_for_scaling])

    # ***FIX***: Ensure the DataFrame columns match the exact features used for training
    # Reindex to the order of 'features' and fill any potentially missing with 0
    # This step is crucial for the scaler
    input_df = input_df.reindex(columns=features, fill_value=0)

    # Handle non-numeric types consistently
    for col in input_df.select_dtypes(include=['object']).columns:
        try:
            # Use the same conversion method as applied to X before splitting
            input_df[col] = pd.to_numeric(input_df[col].astype(str), errors='coerce').fillna(0)
        except Exception as e:
             # Print error but continue, coercing to numeric should handle most issues
             print(f"Warning: Could not convert column {col} to numeric in evaluate_alloy_additions: {e}")
             input_df[col] = pd.to_numeric(input_df[col], errors='coerce').fillna(0)


    # Scale the input using the loaded feature scaler
    # The scaler was fitted on X_train, which contains the 'features' columns in order
    input_scaled = loaded_feature_scaler.transform(input_df) # Use loaded_feature_scaler
    input_tensor = torch.tensor(input_scaled, dtype=torch.float32)

    # Make prediction using the loaded PyTorch model
    with torch.no_grad(): # No gradient calculation needed for evaluation
        predicted_scaled = loaded_model(input_tensor).numpy()[0]

    # Inverse transform the scaled prediction to original scale
    # Use the loaded target_scaler
    predicted_chem = loaded_target_scaler.inverse_transform(predicted_scaled.reshape(1, -1))[0] # Use loaded_target_scaler


    # Calculate the objective (RMSE against target chemistry)
    # Use the target_chem_opt_dict to get target values in the correct order (matching 'target' list)
    target_values = np.array([target_chem_opt_dict.get(col, 0) for col in target])

    # Ensure prediction and target have the same number of elements for comparison
    # The number of elements in predicted_chem should match the number of 'target' columns
    num_elements_to_compare = len(target) # Use the length of the filtered target columns
    predicted_chem_sliced = predicted_chem[:num_elements_to_compare]
    target_values_sliced = target_values[:num_elements_to_compare]

    # Calculate RMSE
    # Add a small epsilon to avoid log(0) if using other metrics, but RMSE is fine with 0
    rmse = np.sqrt(np.mean((predicted_chem_sliced - target_values_sliced)**2))

    return rmse, # Comma for DEAP tuple

# Register the evaluation function with DEAP
toolbox.register("evaluate", evaluate_alloy_additions)

# Run Genetic Algorithm
pop = toolbox.population(n=50) # Increase population size? e.g., n=100
hof = tools.HallOfFame(1)
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("min", np.min)

print("Starting Genetic Algorithm Optimization...")
# Run the GA
# Use HallOfFame as a list so we can access the best individual after the run# Run the GA
# Increase number of generations? e.g., ngen=200
logbook = algorithms.eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=100, stats=stats, halloffame=hof, verbose=False) # Corrected keyword argument to halloffame

print("Genetic Algorithm Optimization Finished.")

# Best solution from GA is in hof[0]
if hof:
    best_individual = hof[0]
else:
    # Handle case where hall of fame might be empty (e.g., ngen=0)
    print("Warning: Hall of Fame is empty. Cannot retrieve best individual.")
    best_individual = None
    optimized_alloy_additions = {}


if best_individual is not None:
    # Map the optimized values back to the alloy column names
    # best_individual is a list of optimized values corresponding to alloy_features_in_model
    optimized_alloy_additions_raw = dict(zip(alloy_features_in_model, best_individual))

    # ***FIX: Ensure all final alloy additions are non-negative by clipping at 0***
    optimized_alloy_additions = {key: max(0.0, value) for key, value in optimized_alloy_additions_raw.items()}


    print("\nOptimized Alloy Additions (TabTransformer + GA):")
    # Display as DataFrame for better readability, including all original alloy_cols
    display_optimized_alloys = {col: optimized_alloy_additions.get(col, 0.0) for col in alloy_cols}
    # Ensure the values are displayed with appropriate precision and handle the index name
    display(pd.DataFrame(display_optimized_alloys, index=["Recommended (kg)"]).T.round(4))


    # Predict the final chemistry for the optimized alloys using the trained model
    # Recreate the input DataFrame using the optimized alloys and base inputs
    optimized_input_dict = base_inputs_opt.copy() # Start with base inputs (median of features)
    # Update with the *clipped* optimized alloy values
    optimized_input_dict.update(optimized_alloy_additions)

    # Create DataFrame with the exact 'features' columns
    optimized_input_df = pd.DataFrame([optimized_input_dict]).reindex(columns=features, fill_value=0)

    # Handle non-numeric types consistently
    for col in optimized_input_df.select_dtypes(include=['object']).columns:
        try:
            optimized_input_df[col] = pd.to_numeric(optimized_input_df[col].astype(str), errors='coerce').fillna(0)
        except Exception as e:
             print(f"Warning: Could not convert column {col} to numeric for final prediction: {e}")
             optimized_input_df[col] = pd.to_numeric(optimized_input_df[col], errors='coerce').fillna(0)


    # Scale the input using the loaded feature scaler
    optimized_input_scaled = loaded_feature_scaler.transform(optimized_input_df) # Use loaded_feature_scaler
    optimized_input_tensor = torch.tensor(optimized_input_scaled, dtype=torch.float32)

    with torch.no_grad():
        predicted_scaled_opt = loaded_model(optimized_input_tensor).numpy()[0]

    # Inverse transform using the loaded target scaler
    predicted_chem_opt = loaded_target_scaler.inverse_transform(predicted_scaled_opt.reshape(1, -1))[0] # Use loaded_target_scaler


    print("\nPredicted Final Chemistry vs Target (using optimized alloys):")
    # Align predicted chemistry with target_cols for comparison
    # Create a dictionary from the predicted_chem_opt array using the target column names
    predicted_chem_dict = dict(zip(target, predicted_chem_opt[:len(target)])) # Use the filtered 'target' list

    # Create comparison DataFrame
    # ***FIX: Use aim_chem for the "Aim" column in the comparison DataFrame***
    chem_comparison_df = pd.DataFrame({
        "Target (Max)": [max_chem.get(col, 0) for col in target],
        "Aim": [aim_chem.get(col, 0) for col in target], # Use aim_chem here
        "Min": [min_chem.get(col, 0) for col in target],
        "Predicted": [predicted_chem_dict.get(col, 0) for col in target]
    }, index=target)

    # Add Delta to Aim column for easier comparison
    chem_comparison_df['Delta to Aim'] = chem_comparison_df['Predicted'] - chem_comparison_df['Aim']


    display(chem_comparison_df.round(5))

    # Optional: Calculate and print RMSE for the predicted vs Aim chemistry for the optimization result
    # Use the values from the comparison DataFrame for the calculation
    aim_values = chem_comparison_df['Aim'].dropna().values
    predicted_values = chem_comparison_df['Predicted'].dropna().values

    # Only calculate RMSE if there are non-NaN values to compare and lengths match
    if len(aim_values) > 0 and len(predicted_values) == len(aim_values):
        rmse_predicted_to_aim = np.sqrt(mean_squared_error(aim_values, predicted_values))
        print(f"\nRMSE of Predicted vs Aim (TabTransformer + GA Optimization): {rmse_predicted_to_aim:.4f}")
    else:
        print("\nCannot calculate RMSE of Predicted vs Aim: Mismatched or missing data for comparison.")


else:
    print("Optimization did not produce a valid best individual.")

# --- End of TabTransformer + GA Optimization Block ---

<ipython-input-125-dc8cce7a0e05>:47: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.ffill() # Use ffill() directly


Numerical TabTransformer model saved to: models/numerical_tab_transformer.pth
Feature scaler saved to: models/feature_scaler.pkl
Target scaler saved to: models/target_scaler.pkl
Test RMSE (Inverse Scaled): 0.0026
Test R² (Inverse Scaled): -2093326925824.0000
Starting Genetic Algorithm Optimization...
Genetic Algorithm Optimization Finished.

Optimized Alloy Additions (TabTransformer + GA):


,Recommended (kg)
CSP-SiMn,0.0000
Mn HC,75.8779
Mn MC,0.0000
Mn LC,0.0000
Mn Metal,0.0000
FeSi,0.0000
Ladle Cov,0.0000
FeMo Metal,0.0000
FeV,0.0000
FeNb lumps,0.0000



Predicted Final Chemistry vs Target (using optimized alloys):


,Target (Max),Aim,Min,Predicted,Delta to Aim
F-C%,0.060,0.050,0.02,0.04220,-0.00780
F-Mn%,0.150,0.120,0.10,0.13416,0.01416
F-S%,0.010,0.000,0.00,0.00556,0.00556
F-P%,0.030,0.015,0.00,0.01275,-0.00225
F-Si%,0.050,0.040,0.00,0.02512,-0.01488
F-Cr%,0.050,0.000,0.00,0.01319,0.01319
F-Ni%,0.050,0.000,0.00,0.00338,0.00338
F-Mo%,0.005,0.000,0.00,0.00040,0.00040
F-V%,0.003,0.000,0.00,0.00058,0.00058
F-Ti%,0.003,0.000,0.00,0.00097,0.00097



RMSE of Predicted vs Aim (TabTransformer + GA Optimization): 0.0071


In [126]:
import pandas as pd
import numpy as np
import random
import joblib
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from deap import base, creator, tools, algorithms
from sklearn.metrics import mean_squared_error
from IPython.display import display
import time # Import time to measure execution

# --- Corrected Preprocessing Functions (as in the previous fix) ---
# Include the corrected definitions of load_data, load_summary, handle_missing,
# create_delta_columns, clip_outliers, preprocess_pipeline, preprocess_data here.
# (Copy-paste the corrected functions from the previous response)

def load_data(filepath):
    # Use os.path.basename to get the filename for checking the extension
    df = pd.read_excel(filepath, sheet_name="Heats") if os.path.basename(filepath).endswith('.xlsx') else pd.read_csv(filepath)
    df.columns = df.columns.str.strip()
    if df.iloc[0].isnull().sum() < 5:
        df.columns = df.iloc[0]
        df = df[1:]
        df.columns = df.columns.str.strip()
    # Avoid inplace=True and chained assignment warnings by chaining calls and assigning back
    # Also add .copy() after dropping rows/columns to avoid SettingWithCopyWarning later
    df = df.dropna(axis=1, how='all').dropna(axis=0, how='all').copy()
    return df

def load_summary(filepath):
    # Use os.path.basename to extract filename from filepath
    if os.path.basename(filepath).endswith('.xlsx'): # Use os.path.basename to extract filename
        summary_df = pd.read_excel(filepath, sheet_name="Summary")
        summary_df.columns = summary_df.columns.str.strip()
        # Avoid inplace=True and add .copy()
        summary_df = summary_df.dropna(how='all').copy()
        return summary_df
    return None

def handle_missing(df):
    # Avoid inplace=True and chained assignment warnings
    # ***FIX: Replace fillna(method='ffill') with ffill()***
    df = df.ffill() # Use ffill() directly
    # Then fill remaining NaNs with the median, specifying numeric_only=True
    # Use the result of fillna directly and assign back to df
    # Add .infer_objects(copy=False) to address downcasting warning
    df = df.fillna(df.median(numeric_only=True)).infer_objects(copy=False)
    return df # Return the modified DataFrame

def create_delta_columns(df):
    # Avoid modifying input df directly with inplace=True and chained assignment warnings
    df_copy = df.copy() # Work on a copy to prevent modifying the original df unexpectedly
    open_chem = ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']
    final_chem = [f"F-{el}" for el in ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']]
    # Ensure columns exist before attempting calculation
    for open_col, final_col in zip(open_chem, final_chem):
        if open_col in df_copy.columns and final_col in df_copy.columns:
            delta_col = f"Delta_{open_col.replace('%', '')}"
            # Ensure columns are numeric before subtraction
            # Use errors='coerce' to turn non-numeric values into NaN
            df_copy[open_col] = pd.to_numeric(df_copy[open_col], errors='coerce')
            df_copy[final_col] = pd.to_numeric(df_copy[final_col], errors='coerce')
            df_copy[delta_col] = df_copy[final_col] - df_copy[open_col]
            # Fill NaNs that might result from coercion or original NaNs
            # Calculate median of the delta column and use fillna on the column directly
            median_delta = df_copy[delta_col].median()
            # Avoid inplace=True
            df_copy[delta_col] = df_copy[delta_col].fillna(median_delta)

    return df_copy # Return the modified copy of the DataFrame

def clip_outliers(df):
    # Avoid modifying input df directly with inplace=True warnings
    df_copy = df.copy() # Work on a copy
    # Select only numeric columns for clipping
    numeric_cols = df_copy.select_dtypes(include=np.number).columns
    for col in numeric_cols:
        # Calculate quantiles and IQR, ignoring NaNs
        q1 = df_copy[col].quantile(0.25)
        q3 = df_copy[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        # Clip values in the column and assign back
        # Avoid inplace=True
        df_copy[col] = df_copy[col].clip(lower=lower_bound, upper=upper_bound)
    return df_copy # Return the modified copy

def preprocess_pipeline(filepath):
    df = load_data(filepath)
    summary_df = load_summary(filepath)
    # Chain the function calls and reassign the result
    # Each function should return a new or modified DataFrame
    df = handle_missing(df)
    df = create_delta_columns(df)
    df = clip_outliers(df)
    # Note: This pipeline returns the processed df and the summary_df separately
    return df, summary_df

def preprocess_data(df):
    # Avoid modifying input df directly
    df = df.copy()
    # Convert datetime or object to numeric where needed
    for col in df.columns:
        if df[col].dtype == 'O':  # If the column is of object (string) type
            try:
                # Specify the format of your date/time column here (if known)
                # Use errors='coerce' and fillna to handle non-datetime strings gracefully
                # Assign the result back to the column
                df[col] = pd.to_datetime(df[col], errors='coerce').dt.total_seconds().fillna(0)
            except Exception: # Catch other potential errors during conversion
                 try:
                      # Handle cases where the column is not a date/time but might be numeric strings
                      # Assign the result back to the column
                      df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
                 except Exception as e:
                      print(f"Could not convert column {col} to numeric in preprocess_data: {e}")
                      # Assign a default value or leave as NaN if conversion fails completely
                      df[col] = np.nan # Assign NaN then the fillna(0) below will handle it


    # Fill remaining NaNs with 0 (as done in your original preprocess_data)
    # Avoid inplace=True
    df = df.fillna(0)

    # Outlier clipping
    # Avoid modifying input df directly
    # Select only numeric columns for clipping
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        # Clip values in the column and assign back
        # Avoid inplace=True
        df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

    return df # Return the modified DataFrame

# Re-define the TabTransformer model class if it's not in scope
# Assuming this is the model you trained
class NumericalTabTransformer(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=128, num_layers=2, num_heads=2):
        super(NumericalTabTransformer, self).__init__()

        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()

        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim), # Another linear layer for transformation
                nn.LayerNorm(hidden_dim) # Add Layer Normalization
            ) for _ in range(num_layers)
        ])

        self.output_layer = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.relu(self.input_layer(x))
        for layer in self.layers:
            x = x + layer(x)
        x = self.output_layer(x)
        return x

# --- End Corrected Preprocessing Functions and Model Class ---


# --- TabTransformer Model Training Block (Ensure this is run first) ---

# 1. Preprocessing and Scaling
filepath = "FE Alloying.xlsx"
df, summary_df = preprocess_pipeline(filepath)

# Define alloy and process features
alloy_cols = [
    "CSP-SiMn", "Mn HC", "Mn MC", "Mn LC", "Mn Metal", "FeSi", "Ladle Cov",
    "FeMo Metal", "FeV", "FeNb lumps", "FeTi lumps", "FeTi Wire", "FeB", "FeAl",
    "Cal Carb", "Al bar", "Al  wire", "FeP", "Sul Stick", "Al mix", "CaSi wire",
    "Cal Wire", "CaFeAl Wire", "S Wire", "Ni Plate", "FeCr LC", "FeCr HC",
    "Al Shot", "Lead Wire", "Mo Metal", "Syn Slag"
]
process_chem_cols_base = [
    'Lift Temp', 'Liquidus temp (° C)', 'Arching Time-mm',
    'LRF Holding Time-mm', 'LRF Lime',
    'C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%'
]
delta_cols = [f"Delta_{el.replace('%','')}" for el in process_chem_cols_base]
features = [col for col in alloy_cols + process_chem_cols_base + delta_cols if col in df.columns]

target_cols = [f"F-{el}" for el in ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']]
target = [col for col in target_cols if col in df.columns]

# Select data using the filtered features and target lists
# ***FIX: Explicitly create a copy of X and y to avoid SettingWithCopyWarning***
X = df[features].copy() # Use .copy() here
y = df[target].copy() # Use .copy() here

# Handle datetime.time columns and other non-numeric types by coercing to numeric
# This step needs to be applied consistently to X and any future input dataframes
for col in X.select_dtypes(include=['object']).columns:
    try:
        # Ensure this conversion method matches the one used later in evaluate_alloy_additions
        # Convert to string first to handle mixed types or datetime.time
        X[col] = pd.to_numeric(X[col].astype(str), errors='coerce').fillna(0)
    except Exception as e:
        print(f"Could not convert column {col} to numeric in X: {e}")


# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Scale target variables separately
target_scaler = StandardScaler()
y_train_scaled = target_scaler.fit_transform(y_train)
y_test_scaled = target_scaler.transform(y_test)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

# Create DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 2. PyTorch TabTransformer Model (adapted for numerical features)
input_dim = X_train_scaled.shape[1]
output_dim = y_train_scaled.shape[1]
model = NumericalTabTransformer(input_dim, output_dim)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. Model Training
num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

# Save the trained PyTorch model and the scalers
os.makedirs("models", exist_ok=True)

model_path = "models/numerical_tab_transformer.pth"
feature_scaler_path = "models/feature_scaler.pkl"
target_scaler_path = "models/target_scaler.pkl"

torch.save(model.state_dict(), model_path)
print(f"Numerical TabTransformer model saved to: {model_path}")

joblib.dump(scaler, feature_scaler_path)
joblib.dump(target_scaler, target_scaler_path)
print(f"Feature scaler saved to: {feature_scaler_path}")
print(f"Target scaler saved to: {target_scaler_path}")


# Model Evaluation (RMSE, R²)
model.eval()
with torch.no_grad():
    predictions = []
    actuals = []
    for inputs, targets in test_loader:
        outputs = model(inputs)
        predictions.append(outputs.numpy())
        actuals.append(targets.numpy())

predictions = np.concatenate(predictions)
actuals = np.concatenate(actuals)

predictions_inv_scaled = target_scaler.inverse_transform(predictions)
actuals_inv_scaled = target_scaler.inverse_transform(actuals)

rmse = np.sqrt(np.mean((predictions_inv_scaled - actuals_inv_scaled)**2))
from sklearn.metrics import r2_score
r2 = r2_score(actuals_inv_scaled, predictions_inv_scaled)

print(f"Test RMSE (Inverse Scaled): {rmse:.4f}")
print(f"Test R² (Inverse Scaled): {r2:.4f}")

# --- End TabTransformer Model Training Block ---


# --- TabTransformer + GA Optimization Block (Modified for multiple runs) ---

def run_ga_optimization(model_path, feature_scaler_path, target_scaler_path, base_inputs, target_chemistry_dict,
                        alloy_elements, all_model_features, df_successful,
                        population_size=50, num_generations=100):
    """
    Runs a single Genetic Algorithm optimization run.

    Args:
        model_path, feature_scaler_path, target_scaler_path (str): Paths to model/scalers.
        base_inputs (dict): Base process and open chemistry inputs.
        target_chemistry_dict (dict): Target final chemistry values (original scale).
        alloy_elements (list): List of alloy elements to optimize.
        all_model_features (list): List of all features the model was trained on.
        df_successful (pd.DataFrame): DataFrame for determining alloy bounds.
        population_size (int): Number of individuals in the GA population.
        num_generations (int): Number of generations to run the GA.

    Returns:
        tuple: A dictionary of optimized alloy additions, the predicted final chemistry
               for these additions, and the final RMSE.
    """
    # Load the trained PyTorch model and scalers
    loaded_model = NumericalTabTransformer(len(all_model_features), len(target_chemistry_dict)) # Match dimensions
    loaded_model.load_state_dict(torch.load(model_path))
    loaded_model.eval()

    loaded_feature_scaler = joblib.load(feature_scaler_path)
    loaded_target_scaler = joblib.load(target_scaler_path)

    # Get base inputs (must match the features the model was trained on)
    # Ensure base_inputs is reindexed to match all_model_features order
    base_inputs_df = pd.DataFrame([base_inputs]).reindex(columns=all_model_features, fill_value=0)
    # Handle non-numeric types consistently as done during training
    for col in base_inputs_df.select_dtypes(include=['object']).columns:
        try:
            base_inputs_df[col] = pd.to_numeric(base_inputs_df[col].astype(str), errors='coerce').fillna(0)
        except Exception as e:
             print(f"Warning: Could not convert column {col} to numeric in base_inputs for GA: {e}")
             base_inputs_df[col] = pd.to_numeric(base_inputs_df[col], errors='coerce').fillna(0)
    base_inputs_reindexed = base_inputs_df.iloc[0].to_dict()


    # Identify the alloy features included in the model's features
    alloy_features_in_model = [col for col in alloy_elements if col in all_model_features]

    # Bounds for alloy elements for optimization
    bounds = [(max(0, df_successful[el].min()) if el in df_successful.columns else 0.0,
               df_successful[el].max() if el in df_successful.columns else 100.0)
              for el in alloy_features_in_model]
    bounds = [(lower, upper) if lower < upper else (lower, lower + 1e-6 if lower == 0 else lower + abs(lower)*1e-6) for lower, upper in bounds]

    low_bounds = [b[0] for b in bounds]
    up_bounds = [b[1] for b in bounds]

    # DEAP setup
    # Create new types unique to this run to avoid conflicts
    try:
        del creator.FitnessMinGA
        del creator.IndividualGA
    except AttributeError:
        pass
    creator.create("FitnessMinGA", base.Fitness, weights=(-1.0,))
    creator.create("IndividualGA", list, fitness=creator.FitnessMinGA)

    toolbox = base.Toolbox()
    toolbox.register("individual", tools.initIterate, creator.IndividualGA,
                     lambda: [random.uniform(low, high) for low, high in bounds])
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("mate", tools.cxBlend, alpha=0.5)
    toolbox.register("mutate", mutGaussianBounded, mu=0, sigma=1, indpb=0.2, low=low_bounds, up=up_bounds)
    toolbox.register("select", tools.selTournament, tournsize=3)

    # Evaluation function for DEAP
    def evaluate_alloy_additions(individual):
        alloy_values = individual
        alloy_dict = dict(zip(alloy_features_in_model, alloy_values))

        input_data = base_inputs_reindexed.copy() # Use the reindexed base inputs
        input_data.update(alloy_dict)

        # Create DataFrame with exact 'all_model_features' columns
        input_df = pd.DataFrame([input_data]).reindex(columns=all_model_features, fill_value=0)

        # Handle non-numeric types consistently
        for col in input_df.select_dtypes(include=['object']).columns:
             try:
                 input_df[col] = pd.to_numeric(input_df[col].astype(str), errors='coerce').fillna(0)
             except Exception as e:
                  print(f"Warning: Could not convert column {col} to numeric in evaluate_alloy_additions: {e}")
                  input_df[col] = pd.to_numeric(input_df[col], errors='coerce').fillna(0)

        # Scale input
        input_scaled = loaded_feature_scaler.transform(input_df)
        input_tensor = torch.tensor(input_scaled, dtype=torch.float32)

        # Predict (scaled)
        with torch.no_grad():
            predicted_scaled = loaded_model(input_tensor).numpy()[0]

        # Inverse transform prediction
        predicted_chem = loaded_target_scaler.inverse_transform(predicted_scaled.reshape(1, -1))[0]

        # Calculate RMSE against target chemistry (original scale)
        # Ensure target values are in the same order as the model's output targets
        target_values = np.array([target_chemistry_dict.get(col, 0) for col in target_chemistry_dict.keys()]) # Use keys from the target dict

        # Ensure prediction and target have same number of elements for comparison
        num_elements_to_compare = min(len(predicted_chem), len(target_values))
        predicted_chem_sliced = predicted_chem[:num_elements_to_compare]
        target_values_sliced = target_values[:num_elements_to_compare]

        rmse = np.sqrt(np.mean((predicted_chem_sliced - target_values_sliced)**2))

        return rmse,

    toolbox.register("evaluate", evaluate_alloy_additions)

    # Run GA
    pop = toolbox.population(n=population_size)
    hof = tools.HallOfFame(1)
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("min", np.min)

    # Run the GA
    algorithms.eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=num_generations, stats=stats, halloffame=hof, verbose=False)

    # Best solution from this run
    if hof:
        best_individual = hof[0]
        final_rmse = best_individual.fitness.values[0]
        # Map optimized values back to alloy column names (including non-optimized as 0)
        optimized_alloy_additions_raw = dict(zip(alloy_features_in_model, best_individual))
        optimized_alloy_additions = {key: max(0.0, value) for key, value in optimized_alloy_additions_raw.items()}
        optimized_alloy_additions_full = {col: optimized_alloy_additions.get(col, 0.0) for col in alloy_elements}

        # Predict final chemistry for the best individual
        optimized_input_dict = base_inputs_reindexed.copy()
        optimized_input_dict.update(optimized_alloy_additions_full) # Use the full dictionary here

        optimized_input_df = pd.DataFrame([optimized_input_dict]).reindex(columns=all_model_features, fill_value=0)
        for col in optimized_input_df.select_dtypes(include=['object']).columns:
             try:
                 optimized_input_df[col] = pd.to_numeric(optimized_input_df[col].astype(str), errors='coerce').fillna(0)
             except Exception as e:
                  print(f"Warning: Could not convert column {col} to numeric for final prediction: {e}")
                  optimized_input_df[col] = pd.to_numeric(optimized_input_df[col], errors='coerce').fillna(0)

        optimized_input_scaled = loaded_feature_scaler.transform(optimized_input_df)
        optimized_input_tensor = torch.tensor(optimized_input_scaled, dtype=torch.float32)

        with torch.no_grad():
            predicted_scaled_opt = loaded_model(optimized_input_tensor).numpy()[0]

        predicted_chem_opt = loaded_target_scaler.inverse_transform(predicted_scaled_opt.reshape(1, -1))[0]

        return optimized_alloy_additions_full, predicted_chem_opt, final_rmse

    else:
        return None, None, float('inf') # Return None and infinity if no solution found


# --- Run Multiple GA Optimizations ---

# Load necessary data and define parameters (ensure these match the training block)
# Use base_inputs_opt and the target chemistry dictionary you chose (e.g., max_chem or aim_chem)
# These variables should be available from the execution of the training block above.

# Assuming base_inputs_opt, max_chem, aim_chem, min_chem, alloy_cols, features, df_successful are defined

# ***CHOOSE YOUR TARGET CHEMISTRY FOR OPTIMIZATION***
# target_chemistry_for_optimization = max_chem # Or
target_chemistry_for_optimization = aim_chem # ***Often you want to optimize towards Aim***


num_ga_runs = 5 # Number of times to run the GA
population_size = 100 # GA population size
num_generations = 200 # Number of GA generations

best_rmse_across_runs = float('inf')
best_alloys_overall = None
best_predicted_chem_overall = None

all_run_results = [] # Store results from each run

print(f"\n--- Running {num_ga_runs} Genetic Algorithm Optimizations ---")

for run in range(num_ga_runs):
    print(f"Starting GA Run {run + 1}/{num_ga_runs}...")
    start_time = time.time()

    optimized_alloys_this_run, predicted_chem_this_run, rmse_this_run = run_ga_optimization(
        model_path=model_path, # Path from training block
        feature_scaler_path=feature_scaler_path, # Path from training block
        target_scaler_path=target_scaler_path, # Path from training block
        base_inputs=base_inputs_opt, # Use the base inputs dictionary
        target_chemistry_dict=target_chemistry_for_optimization, # Use the chosen target chemistry
        alloy_elements=alloy_cols, # Use the full list of possible alloy columns
        all_model_features=features, # Use the exact list of features the model was trained on
        df_successful=df_successful, # Pass df_successful for bounds
        population_size=population_size,
        num_generations=num_generations
    )

    end_time = time.time()
    print(f"Run {run + 1} finished in {end_time - start_time:.2f} seconds with RMSE: {rmse_this_run:.4f}")

    if optimized_alloys_this_run is not None:
         all_run_results.append({
             'run': run + 1,
             'rmse': rmse_this_run,
             'alloys': optimized_alloys_this_run,
             'predicted_chem': predicted_chem_this_run
         })

        # Track the best result across all runs
         if rmse_this_run < best_rmse_across_runs:
             best_rmse_across_runs = rmse_this_run
             best_alloys_overall = optimized_alloys_this_run
             best_predicted_chem_overall = predicted_chem_this_run

print("\n--- Optimization Complete ---")

# --- Display Results ---

if best_alloys_overall is not None:
    print("\n--- Best Optimized Alloy Additions Across All Runs (TabTransformer + GA) ---")
    # Ensure the optimized_alloys_pso dictionary includes all original alloy_cols, setting non-optimized to 0
    best_alloys_df = pd.DataFrame([best_alloys_overall]).T.rename(columns={0: "Recommended kg"})
    display(best_alloys_df.round(4))

    print("\n--- Predicted Final Chemistry (Best Run) vs Target ---")
    # Ensure consistent keys for comparison using the keys from target_chemistry_for_optimization
    all_target_chem_keys = list(target_chemistry_for_optimization.keys())

    predicted_chem_comparison_data = {
        "Target": [target_chemistry_for_optimization.get(k, np.nan) for k in all_target_chem_keys],
        # Create a dictionary from the predicted array using the target keys for alignment
        "Predicted": dict(zip(all_target_chem_keys, best_predicted_chem_overall[:len(all_target_chem_keys)]))
    }

    chem_comparison_df = pd.DataFrame(predicted_chem_comparison_data, index=all_target_chem_keys)

    # Add Delta to Target column for easier comparison
    chem_comparison_df['Delta to Target'] = chem_comparison_df['Predicted'] - chem_comparison_df['Target']

    display(chem_comparison_df.round(5))

    print(f"\nOverall Best RMSE to Target: {best_rmse_across_runs:.4f}")


    # Optional: You could also calculate the average or median of the optimized alloys
    # across all runs with reasonable RMSE if you prefer an ensemble recommendation.
    # Filter successful runs (e.g., RMSE below a threshold or top N)
    # successful_runs = [res for res in all_run_results if res['rmse'] < best_rmse_across_runs * 1.1] # Example threshold
    # if successful_runs:
    #     # Create a DataFrame from alloy dictionaries
    #     alloy_dfs = [pd.DataFrame([run['alloys']]) for run in successful_runs]
    #     combined_alloys = pd.concat(alloy_dfs, ignore_index=True)
    #     # Calculate median recommendations
    #     median_alloys = combined_alloys.median().to_dict()
    #     print("\n--- Median Recommended Alloy Additions (Across Successful Runs) ---")
    #     display(pd.DataFrame([median_alloys]).T.rename(columns={0: "Recommended kg"}).round(4))

else:
    print("No successful optimization runs completed.")

<ipython-input-126-ea4a4a4779f1>:47: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.ffill() # Use ffill() directly


Numerical TabTransformer model saved to: models/numerical_tab_transformer.pth
Feature scaler saved to: models/feature_scaler.pkl
Target scaler saved to: models/target_scaler.pkl
Test RMSE (Inverse Scaled): 0.0011
Test R² (Inverse Scaled): -40525127680.0000

--- Running 5 Genetic Algorithm Optimizations ---
Starting GA Run 1/5...
Run 1 finished in 47.96 seconds with RMSE: 0.0073
Starting GA Run 2/5...
Run 2 finished in 47.33 seconds with RMSE: 0.0073
Starting GA Run 3/5...
Run 3 finished in 48.36 seconds with RMSE: 0.0073
Starting GA Run 4/5...
Run 4 finished in 47.74 seconds with RMSE: 0.0073
Starting GA Run 5/5...
Run 5 finished in 46.78 seconds with RMSE: 0.0073

--- Optimization Complete ---

--- Best Optimized Alloy Additions Across All Runs (TabTransformer + GA) ---


,Recommended kg
CSP-SiMn,0.0000
Mn HC,0.0000
Mn MC,0.0000
Mn LC,0.0000
Mn Metal,0.0000
FeSi,0.0000
Ladle Cov,0.0000
FeMo Metal,0.0000
FeV,0.0000
FeNb lumps,0.0000



--- Predicted Final Chemistry (Best Run) vs Target ---


,Target,Predicted,Delta to Target
F-C%,0.050,0.04206,-0.00793
F-Mn%,0.120,0.13544,0.01544
F-S%,0.000,0.00664,0.00664
F-P%,0.015,0.01301,-0.00199
F-Si%,0.040,0.02588,-0.01412
F-Cr%,0.000,0.01301,0.01301
F-Ni%,0.000,0.00340,0.00340
F-Mo%,0.000,0.00040,0.00040
F-V%,0.000,-0.00141,-0.00141
F-Ti%,0.000,0.00097,0.00097



Overall Best RMSE to Target: 0.0073


In [112]:
!pip install transformers

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
import tensorflow as tf
import os  # Import the os module
from IPython import get_ipython
from IPython.display import display
import datetime # Import the datetime module

# 1. Load and preprocess your data
# Assuming 'filepath' is defined and points to your Excel file
df, summary_df = preprocess_pipeline(filepath)

# Define alloy and process features
alloy_cols = [
    "CSP-SiMn", "Mn HC", "Mn MC", "Mn LC", "Mn Metal", "FeSi", "Ladle Cov",
    "FeMo Metal", "FeV", "FeNb lumps", "FeTi lumps", "FeTi Wire", "FeB", "FeAl",
    "Cal Carb", "Al bar", "Al  wire", "FeP", "Sul Stick", "Al mix", "CaSi wire",
    "Cal Wire", "CaFeAl Wire", "S Wire", "Ni Plate", "FeCr LC", "FeCr HC",
    "Al Shot", "Lead Wire", "Mo Metal", "Syn Slag"
]
process_chem_cols = [
    'Lift Temp', 'Liquidus temp (° C)', 'Arching Time-mm',
    'LRF Holding Time-mm', 'LRF Lime',
    'C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%'
]
target_cols = [f"F-{el}" for el in ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']]

# Select features and target
features = alloy_cols + process_chem_cols  # Use existing alloy and process features
target = target_cols  # Use final chemistry as target

X = df[features]
y = df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# *** Convert datetime.time columns to numerical representation ***
for col in X_train.select_dtypes(include=['object']).columns:
    if X_train[col].apply(lambda x: isinstance(x, (datetime.time))).any():
        X_train[col] = pd.to_numeric(X_train[col].astype(str), errors='coerce').fillna(0)
        X_test[col] = pd.to_numeric(X_test[col].astype(str), errors='coerce').fillna(0)

# 2. Build the Transformer model (adapted for numerical features)
# For numerical features, we'll skip the tokenizer and use a simple input layer
# We'll also use a MultiOutputRegressor for multiple target properties

def build_transformer_model(input_shape):
    input_layer = tf.keras.layers.Input(shape=input_shape, dtype=tf.float32, name="numerical_features")

    # Added layers:
    x = tf.keras.layers.BatchNormalization()(input_layer)  # Batch Normalization for feature scaling
    x = tf.keras.layers.Dense(64, activation='relu')(x)     # Dense layer with ReLU activation
    x = tf.keras.layers.Dense(32, activation='relu')(x)     # Another Dense layer with ReLU activation

    output_layer = tf.keras.layers.Dense(len(target_cols), activation='linear')(x) # Output layer for multiple targets
    model = tf.keras.Model(inputs=input_layer, outputs=output_layer)
    return model

# 3. Train the model
input_shape = (len(features),)
model = build_transformer_model(input_shape)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5), loss='mse', metrics=['mae'])
model.fit(X_train, y_train, epochs=3, batch_size=32, validation_split=0.1)

# 4. Evaluate the model
loss, mae = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {loss:.4f}, Test MAE: {mae:.4f}")

# 5. Save and use the model for predictions
model_dir = 'models'  # Define a directory for saving the model

# Create the directory if it doesn't exist
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

# Provide a filename with the .keras extension
model_path = os.path.join(model_dir, 'transformer_model.keras')
model.save(model_path)  # Save the model to the specified path with filename

<ipython-input-111-6ad4514ae9f8>:44: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
<ipython-input-111-6ad4514ae9f8>:44: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(method='ffill', inplace=True)
<ipython-input-111-6ad4514ae9f8>:63: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=Tru

Epoch 1/3
73/73 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0391 - mae: 0.1528 - val_loss: 1488.7097 - val_mae: 31.8378
Epoch 2/3
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0328 - mae: 0.1391 - val_loss: 596.7932 - val_mae: 19.4157
Epoch 3/3
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0284 - mae: 0.1300 - val_loss: 262.0151 - val_mae: 12.5302
Test loss: 261.7646, Test MAE: 12.5245


In [124]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
import os  # Import the os module
from IPython import get_ipython
from IPython.display import display
import datetime # Import the datetime module

# 1. Load and preprocess your data
# Assuming 'filepath' is defined and points to your Excel file
df, summary_df = preprocess_pipeline(filepath)

# Define alloy and process features
alloy_cols = [
    "CSP-SiMn", "Mn HC", "Mn MC", "Mn LC", "Mn Metal", "FeSi", "Ladle Cov",
    "FeMo Metal", "FeV", "FeNb lumps", "FeTi lumps", "FeTi Wire", "FeB", "FeAl",
    "Cal Carb", "Al bar", "Al  wire", "FeP", "Sul Stick", "Al mix", "CaSi wire",
    "Cal Wire", "CaFeAl Wire", "S Wire", "Ni Plate", "FeCr LC", "FeCr HC",
    "Al Shot", "Lead Wire", "Mo Metal", "Syn Slag"
]
process_chem_cols = [
    'Lift Temp', 'Liquidus temp (° C)', 'Arching Time-mm',
    'LRF Holding Time-mm', 'LRF Lime',
    'C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%'
]
target_cols = [f"F-{el}" for el in ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']]

# Select features and target
features = alloy_cols + process_chem_cols  # Use existing alloy and process features
target = target_cols  # Use final chemistry as target

X = df[features]
y = df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# *** Convert datetime.time columns to numerical representation ***
for col in X_train.select_dtypes(include=['object']).columns:
    if X_train[col].apply(lambda x: isinstance(x, (datetime.time))).any():
        X_train[col] = pd.to_numeric(X_train[col].astype(str), errors='coerce').fillna(0)
        X_test[col] = pd.to_numeric(X_test[col].astype(str), errors='coerce').fillna(0)

# 2. Build the FCN model
def build_fcn_model(input_shape, num_outputs):
    input_layer = tf.keras.layers.Input(shape=input_shape, dtype=tf.float32, name="numerical_features")

    # Hidden layers
    x = tf.keras.layers.Dense(128, activation='relu')(input_layer)
    x = tf.keras.layers.Dense(64, activation='relu')(x)

    # Output layer (with multiple outputs if needed)
    output_layer = tf.keras.layers.Dense(num_outputs, activation='linear')(x)

    model = tf.keras.Model(inputs=input_layer, outputs=output_layer)
    return model

# 3. Train the model
input_shape = (len(features),)
num_outputs = len(target_cols)  # Number of target properties
model = build_fcn_model(input_shape, num_outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='mse', metrics=['mae']) # Adjusted learning rate
model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.1) # Increased epochs

# 4. Evaluate the model
loss, mae = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {loss:.4f}, Test MAE: {mae:.4f}")

# 5. Save and use the model for predictions
model_dir = 'models'  # Define a directory for saving the model

# Create the directory if it doesn't exist
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

# Provide a filename with the .keras extension
model_path = os.path.join(model_dir, 'my_fcn_model.keras')
model.save(model_path)  # Save the model to the specified path with filename

<ipython-input-123-31ca2f913be7>:47: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.ffill() # Use ffill() directly


Epoch 1/50
73/73 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 19524.3379 - mae: 89.0131 - val_loss: 57.8118 - val_mae: 5.5585
Epoch 2/50
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 44.5601 - mae: 4.6623 - val_loss: 20.0240 - val_mae: 3.1888
Epoch 3/50
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 19.0289 - mae: 3.0800 - val_loss: 13.9626 - val_mae: 2.6786
Epoch 4/50
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 13.3391 - mae: 2.6029 - val_loss: 9.3840 - val_mae: 2.1900
Epoch 5/50
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 9.4430 - mae: 2.1765 - val_loss: 7.1694 - val_mae: 1.9456
Epoch 6/50
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 6.8008 - mae: 1.8629 - val_loss: 5.0016 - val_mae: 1.5950
Epoch 7/50
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.8689 - mae: 1.5756 - val_loss: 3.8140 - val_mae: 1.3921
Epoch 8/50
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3.6804 - mae: 1.3720 - val_loss: 2.9852 - val_mae: 1.2449
Epoch 9/50
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.

In [ ]:
# Preprocess
df, summary_df = preprocess_pipeline(filepath)

# Extract alloy and process features
alloy_cols = [
    "CSP-SiMn", "Mn HC", "Mn MC", "Mn LC", "Mn Metal", "FeSi", "Ladle Cov",
    "FeMo Metal", "FeV", "FeNb lumps", "FeTi lumps", "FeTi Wire", "FeB", "FeAl",
    "Cal Carb", "Al bar", "Al  wire", "FeP", "Sul Stick", "Al mix", "CaSi wire",
    "Cal Wire", "CaFeAl Wire", "S Wire", "Ni Plate", "FeCr LC", "FeCr HC",
    "Al Shot", "Lead Wire", "Mo Metal", "Syn Slag"
]

process_chem_cols = [
    'Lift Temp', 'Liquidus temp (° C)', 'Arching Time-mm',
    'LRF Holding Time-mm', 'LRF Lime',
    'C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%'
]

# Extract successful heats
max_summary_row = summary_df.iloc[2].fillna(0)
aim_summary_row = summary_df.iloc[3].fillna(0)
min_summary_row = summary_df.iloc[1].fillna(0)
print(aim_summary_row)
target_keys = [k.strip() for k in max_summary_row.index if k.strip().endswith("%")]
target_vector = max_summary_row[target_keys].dropna().astype(float)
present_f_cols = [f"F-{k}" for k in target_keys if f"F-{k}" in df.columns]
df['Success_Score'] = -((df[present_f_cols] - target_vector.loc[[k for k in target_keys if f"F-{k}" in df.columns]].values) ** 2).sum(axis=1) ** 0.3
df_successful = df[df['Success_Score'] >= df['Success_Score'].quantile(0.5)]


# Get target chemical keys (elements with %)
# target_keys = [k.strip() for k in max_summary_row.index if k.strip().endswith("%")]

# # Create a mask for rows within min-max range for each F-chemical
# mask = pd.Series(True, index=df.index)  # Initialize mask to True for all rows

# 1. Attempt filtering with min-max range
# mask = pd.Series(True, index=df.index)
# for element in target_keys:
#     f_col = f"F-{element}"
#     if f_col in df.columns:
#         min_val = min_summary_row.get(element, -np.inf)
#         max_val = max_summary_row.get(element, np.inf)
#         mask &= (df[f_col] >= min_val) & (df[f_col] <= max_val)

# df_successful = df[mask]

# # 2. If df_successful is empty, consider aim closer values
# if df_successful.empty:
#     df['Success_Score'] = 0  # Initialize Success_Score column
#     for element in target_keys:
#         f_col = f"F-{element}"
#         if f_col in df.columns:
#             aim_val = aim_summary_row.get(element, 0)  # Get aim value
#             df['Success_Score'] += (df[f_col] - aim_val) ** 2  # Calculate squared difference

#     df['Success_Score'] = df['Success_Score'] ** 0.5  # Take square root for overall distance
#     df_successful = df[df['Success_Score'] <= df['Success_Score'].quantile(0.5)]  # Select closest 50%


print(df_successful.shape)  # Check if it's empty
print(df_successful[alloy_cols].describe())  # Check summary statistics

# Select base inputs from median of successful heats
selected_cols = [col for col in process_chem_cols + alloy_cols if col in df_successful.columns]
base_inputs = df_successful[selected_cols].median(numeric_only=True).to_dict()

# Extract target chemistry
max_chem = {
    f"F-{k.strip()}": float(v) for k, v in max_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v)
}

aim_chem = {
    f"F-{k.strip()}": float(v) for k, v in aim_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v)
}

min_chem = {
    f"F-{k.strip()}": float(v) for k, v in min_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v)
}

# Define the model directory and filename
model_dir = 'models'
model_filename = 'my_fcn_model.keras' # Ensure this matches the save filename
model_path = os.path.join(model_dir, model_filename)


# Ensure model is loaded if it wasn't in the previous execution
try:
    # Correct the path to load the model from the 'models' directory
    model = tf.keras.models.load_model(model_path)
    print("Keras model loaded successfully.")
except FileNotFoundError:
    # Provide a more informative error message if the file is not found
    print(f"Error: Model file not found at {model_path}. Please make sure the cell that saves the model has been run.")
    model = None # Set model to None if not found


if model is not None:
    # Assuming you want to optimize towards max_chem
    # Pass the feature names to the optimization function
    # You can get feature names from the columns of your training data (e.g., X_train)
    # Since X was created using 'features', use that list here.
    feature_names = alloy_cols + process_chem_cols
    # Define target_cols before calling the function
    target_cols = [f"F-{el}" for el in ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']]

    optimized_alloys_ga, predicted_chem_ga = optimize_alloy_additions_tf(
        model=model,
        base_inputs=base_inputs,
        target_chemistry=max_chem,
        alloy_elements=alloy_cols,
        df_successful=df_successful,
        feature_names=feature_names # Pass the feature names here

    )

    print("\n--- GA Optimization Results ---")
    print("Optimized Alloy Additions (GA):")
    print(pd.DataFrame(optimized_alloys_ga, index=["Recommended (kg)"]))

    # The predicted_chem_ga from the GA objective function is an array matching the
    # order of model outputs. To display it meaningfully, we need the corresponding
    # target chemistry column names. We can get these from the target_chem dictionary keys.
    # Use target_cols to align the prediction output with the column names
    predicted_chem_dict = dict(zip(target_cols, predicted_chem_ga[:len(target_cols)]))


    print("\nPredicted Final Chemistry vs Target (GA):")
    chem_df_ga = pd.DataFrame({
        "Target": target_chem, # Using the selected target chemistry
        "Predicted": predicted_chem_dict # Use the dictionary with keys
    })
    print(chem_df_ga.round(5))

In [ ]:
# ipython-input-73-ba18fefa979b (Corrected)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
import os  # Import the os module
from IPython import get_ipython
from IPython.display import display
import datetime # Import the datetime module
from deap import base, creator, tools, algorithms # Import DEAP components
# Import the losses module from Keras
from tensorflow.keras import losses


# Helper function to mutate within bounds
def mutate_within_bounds_tf(individual, mu, sigma, indpb, bounds):
    """Mutates an individual and clips values to stay within bounds."""
    # Apply standard Gaussian mutation
    tools.mutGaussian(individual, mu, sigma, indpb)
    # Clip mutated values to stay within bounds
    for i, (low, up) in enumerate(bounds):
        individual[i] = np.clip(individual[i], low, up)
    return individual,

def optimize_alloy_additions_tf(model, base_inputs, target_chemistry, alloy_elements, df_successful, feature_names):
    """
    Optimizes alloy additions using a trained TensorFlow/Keras model with DEAP.

    Args:
        model (tf.keras.Model): The trained TensorFlow/Keras model.
        base_inputs (dict): Dictionary of base process and open chemistry inputs.
        target_chemistry (dict): Dictionary of target final chemistry values.
        alloy_elements (list): List of alloy elements to optimize.
        df_successful (pd.DataFrame): DataFrame of successful heats for determining alloy bounds.
        feature_names (list): List of input feature names the model expects, in order.

    Returns:
        tuple: A dictionary of optimized alloy additions and the predicted final chemistry.
    """
    # Bounds for each alloy based on min/max scaling in successful heats
    # Ensure bounds are non-negative
    # Check if the alloy column exists in df_successful before getting min/max
    bounds = [
        (max(0.0, df_successful[el].min()) if el in df_successful.columns else 0.0,
         df_successful[el].max() if el in df_successful.columns else 100.0) # Provide a default upper bound if column missing
        for el in alloy_elements
    ]
    # Further refine bounds to ensure lower is less than or equal to upper
    bounds = [(low, high) if low <= high else (low, low + 1e-6) for low, high in bounds]


    print("Alloy Bounds for TF Optimization:")
    for i, el in enumerate(alloy_elements):
        print(f"{el}: Min = {bounds[i][0]:.2f}, Max = {bounds[i][1]:.4f}")

    # Prepare input for model prediction
    def prepare_input(alloy_values):
        input_dict = base_inputs.copy()
        input_dict.update(dict(zip(alloy_elements, alloy_values)))

        # Ensure the order of columns matches model's expected input features
        input_df = pd.DataFrame([input_dict]).reindex(columns=feature_names, fill_value=0)

        # Convert to a format the Keras model expects (e.g., numpy array)
        # Handle potential datetime objects from base_inputs if they weren't fully converted
        # Convert any object columns to numeric, coercing errors to NaN, then fill NaN with 0
        for col in input_df.select_dtypes(include=['object']).columns:
            try:
                 # Attempt to convert datetime objects if present
                 if input_df[col].apply(lambda x: isinstance(x, (datetime.time, pd.Timestamp))).any():
                     input_df[col] = pd.to_numeric(input_df[col].astype(str), errors='coerce')
            except:
                 pass # Ignore if not datetime

            # Convert remaining object columns to numeric, coercing errors
            input_df[col] = pd.to_numeric(input_df[col], errors='coerce')

        # Fill any remaining NaNs (e.g., from coerce or reindex)
        input_df = input_df.fillna(0)

        return input_df.values # Return as numpy array

    def fitness_function(individual):
        input_array = prepare_input(individual)
        # Predict using the Keras model; prediction is a numpy array
        prediction = model.predict(input_array)[0]

        # Get output feature names (target chemistry elements) from target_chemistry keys
        output_feature_names = list(target_chemistry.keys()) # Use the keys provided as target_chemistry

        # Create target array in the order of model output features
        # Ensure target has the same order as prediction based on output_feature_names
        # Take target values only for the elements the model predicts (or target_chemistry provides)
        # The model output layer has `len(target_cols)` neurons, which should correspond
        # to the number of keys in target_chemistry if target_chemistry was properly defined
        # during model training setup.
        target = np.array([target_chemistry.get(col, 0) for col in output_feature_names])


        # Ensure prediction and target have same number of elements for comparison
        # This handles cases where the model might predict fewer outputs than in target_chemistry
        num_elements_to_compare = min(len(prediction), len(target))
        prediction_subset = prediction[:num_elements_to_compare]
        target_subset = target[:num_elements_to_compare]

        # Calculate RMSE using tf.keras.losses.MeanSquaredError
        # Use the functional form which takes y_true and y_pred directly
        mse = losses.MeanSquaredError()(target_subset, prediction_subset).numpy()

        return np.sqrt(mse), # Comma for DEAP tuple


    # --- DEAP setup ---
    # Reset creator definitions to allow running multiple times in a notebook
    try:
        del creator.FitnessMinTF
        del creator.IndividualTF
    except AttributeError:
        pass # Objects not created yet

    # Create new types for this run
    creator.create("FitnessMinTF", base.Fitness, weights=(-1.0,))
    creator.create("IndividualTF", list, fitness=creator.FitnessMinTF)

    # Create a new toolbox for this optimization run
    toolbox = base.Toolbox()

    # Register tools for this toolbox instance
    toolbox.register("attr_float", lambda i: np.random.uniform(bounds[i][0], bounds[i][1]))
    toolbox.register("individual", tools.initIterate, creator.IndividualTF,
                     lambda: [np.random.uniform(low, high) for low, high in bounds])
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", fitness_function)
    toolbox.register("mate", tools.cxBlend, alpha=0.5)
    # Register the custom mutation function that handles bounds
    toolbox.register("mutate", mutate_within_bounds_tf, mu=0, sigma=1, indpb=0.2, bounds=bounds)

    toolbox.register("select", tools.selTournament, tournsize=3)


    # Run GA
    pop = toolbox.population(n=30)
    hof = tools.HallOfFame(1)

    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("min", np.min)

    algorithms.eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=20, stats=stats, halloffame=hof, verbose=True)

    # Best solution
    best_individual = hof[0]
    best_prediction = model.predict(prepare_input(best_individual))[0]

    # Return optimized alloy additions and the predicted final chemistry
    return dict(zip(alloy_elements, best_individual)), best_prediction

# Preprocess
df, summary_df = preprocess_pipeline(filepath)

# Extract alloy and process features
alloy_cols = [
    "CSP-SiMn", "Mn HC", "Mn MC", "Mn LC", "Mn Metal", "FeSi", "Ladle Cov",
    "FeMo Metal", "FeV", "FeNb lumps", "FeTi lumps", "FeTi Wire", "FeB", "FeAl",
    "Cal Carb", "Al bar", "Al  wire", "FeP", "Sul Stick", "Al mix", "CaSi wire",
    "Cal Wire", "CaFeAl Wire", "S Wire", "Ni Plate", "FeCr LC", "FeCr HC",
    "Al Shot", "Lead Wire", "Mo Metal", "Syn Slag"
]

process_chem_cols = [
    'Lift Temp', 'Liquidus temp (° C)', 'Arching Time-mm',
    'LRF Holding Time-mm', 'LRF Lime',
    'C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%'
]

# Extract successful heats
max_summary_row = summary_df.iloc[2].fillna(0)
aim_summary_row = summary_df.iloc[3].fillna(0)
min_summary_row = summary_df.iloc[1].fillna(0)
print(aim_summary_row)
target_keys = [k.strip() for k in max_summary_row.index if k.strip().endswith("%")]
target_vector = max_summary_row[target_keys].dropna().astype(float)
present_f_cols = [f"F-{k}" for k in target_keys if f"F-{k}" in df.columns]
df['Success_Score'] = -((df[present_f_cols] - target_vector.loc[[k for k in target_keys if f"F-{k}" in df.columns]].values) ** 2).sum(axis=1) ** 0.3
df_successful = df[df['Success_Score'] >= df['Success_Score'].quantile(0.5)].copy() # Use .copy()


# Get target chemical keys (elements with %)
# target_keys = [k.strip() for k in max_summary_row.index if k.strip().endswith("%")]

# # Create a mask for rows within min-max range for each F-chemical
# mask = pd.Series(True, index=df.index)  # Initialize mask to True for all rows

# 1. Attempt filtering with min-max range
# mask = pd.Series(True, index=df.index)
# for element in target_keys:
#     f_col = f"F-{element}"
#     if f_col in df.columns:
#         min_val = min_summary_row.get(element, -np.inf)
#         max_val = max_summary_row.get(element, np.inf)
#         mask &= (df[f_col] >= min_val) & (df[f_col] <= max_val)

# df_successful = df[mask]

# # 2. If df_successful is empty, consider aim closer values
# if df_successful.empty:
#     df['Success_Score'] = 0  # Initialize Success_Score column
#     for element in target_keys:
#         f_col = f"F-{element}"
#         if f_col in df.columns:
#             aim_val = aim_summary_row.get(element, 0)  # Get aim value
#             df['Success_Score'] += (df[f_col] - aim_val) ** 2  # Calculate squared difference

#     df['Success_Score'] = df['Success_Score'] ** 0.5  # Take square root for overall distance
#     df_successful = df[df['Success_Score'] <= df['Success_Score'].quantile(0.5)]  # Select closest 50%


print(df_successful.shape)  # Check if it's empty
print(df_successful[alloy_cols].describe())  # Check summary statistics

# Select base inputs from median of successful heats
selected_cols = [col for col in process_chem_cols + alloy_cols if col in df_successful.columns]
base_inputs = df_successful[selected_cols].median(numeric_only=True).to_dict()

# Extract target chemistry
max_chem = {
    f"F-{k.strip()}": float(v) for k, v in max_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v)
}

aim_chem = {
    f"F-{k.strip()}": float(v) for k, v in aim_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v)
}

min_chem = {
    f"F-{k.strip()}": float(v) for k, v in min_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v)
}

# Define the model directory and filename
model_dir = 'models'
model_filename = 'my_fcn_model.keras' # Ensure this matches the save filename
model_path = os.path.join(model_dir, model_filename)


# Ensure model is loaded if it wasn't in the previous execution
try:
    # Correct the path to load the model from the 'models' directory
    model = tf.keras.models.load_model(model_path)
    print("Keras model loaded successfully.")
except FileNotFoundError:
    # Provide a more informative error message if the file is not found
    print(f"Error: Model file not found at {model_path}. Please make sure the cell that saves the model has been run.")
    model = None # Set model to None if not found


if model is not None:
    # Assuming you want to optimize towards max_chem
    # Pass the feature names to the optimization function
    # You can get feature names from the columns of your training data (e.g., X_train)
    # Since X was created using 'features', use that list here.
    feature_names = alloy_cols + process_chem_cols
    # Define target_cols before calling the function
    target_cols = [f"F-{el}" for el in ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']]

    optimized_alloys_ga, predicted_chem_ga = optimize_alloy_additions_tf(
        model=model,
        base_inputs=base_inputs,
        target_chemistry=max_chem, # Use the desired target chemistry (e.g., max_chem)
        alloy_elements=alloy_cols,
        df_successful=df_successful,
        feature_names=feature_names # Pass the feature names here
    )

    print("\n--- GA Optimization Results ---")
    print("Optimized Alloy Additions (GA):")
    print(pd.DataFrame(optimized_alloys_ga, index=["Recommended (kg)"]))

    # The predicted_chem_ga from the GA objective function is an array matching the
    # order of model outputs. To display it meaningfully, we need the corresponding
    # target chemistry column names. We can get these from the target_chem dictionary keys.
    # Use target_cols to align the prediction output with the column names
    predicted_chem_dict = dict(zip(target_cols, predicted_chem_ga[:len(target_cols)]))


    print("\nPredicted Final Chemistry vs Target (GA):")
    # Use max_chem keys for the target column names in the comparison DataFrame,
    # as max_chem was used as the target for optimization.
    comparison_target_keys = list(max_chem.keys())
    # Ensure predicted_chem_dict only contains keys that are also in comparison_target_keys
    predicted_chem_comparison = {k: predicted_chem_dict.get(k, np.nan) for k in comparison_target_keys}


    chem_df_ga = pd.DataFrame({
        "Target (Max)": [max_chem.get(k, np.nan) for k in comparison_target_keys],
        "Predicted": [predicted_chem_comparison.get(k, np.nan) for k in comparison_target_keys]
    }, index=comparison_target_keys) # Set index to the chemistry elements

    print(chem_df_ga.round(5))

In [ ]:
import numpy as np
import pandas as pd
from skopt import gp_minimize
import tensorflow as tf
import os
import datetime
from IPython.display import display

# Assuming these variables are defined and populated in previous cells:
# df_successful (pd.DataFrame): DataFrame containing successful heats.
# base_inputs (dict): Dictionary of median process and initial chemistry values from successful heats.
# max_chem (dict): Dictionary of maximum target final chemistry values.
# aim_chem (dict): Dictionary of aim target final chemistry values.
# min_chem (dict): Dictionary of minimum target final chemistry values.
# alloy_cols (list): List of alloy column names.
# process_chem_cols (list): List of process and initial chemistry column names.
# target_cols (list): List of target final chemistry column names.
# model_transformer (tf.keras.Model): Your loaded Keras Transformer model.


def run_bayesian_optimization_transformer(model_transformer, df_successful, base_inputs, max_chem, aim_chem, min_chem, alloy_cols, process_chem_cols, target_cols, n_calls=50):
    """
    Runs Bayesian Optimization to find optimal alloy additions
    using a trained Keras Transformer model.

    Args:
        model_transformer (tf.keras.Model): Your loaded Keras Transformer model.
        df_successful (pd.DataFrame): DataFrame containing successful heats (used for bounds).
        base_inputs (dict): Dictionary of median process and initial chemistry values from successful heats.
        max_chem (dict): Dictionary of maximum target final chemistry values (optimization target).
        aim_chem (dict): Dictionary of aim target final chemistry values (for comparison).
        min_chem (dict): Dictionary of minimum target final chemistry values (for comparison).
        alloy_cols (list): List of alloy column names.
        process_chem_cols (list): List of process and initial chemistry column names.
        target_cols (list): List of target final chemistry column names (for comparison and order).
        n_calls (int): Number of calls to the objective function (optimization iterations).

    Returns:
        tuple: A tuple containing:
            - optimized_alloys (dict): Dictionary of optimized alloy additions (kg).
            - predicted_chem (np.ndarray): Array of predicted final chemistry values.
            - comparison_df (pd.DataFrame): DataFrame comparing predicted vs target chemistry.
    """

    # --- Placeholder function for preparing Transformer input ---
    # YOU NEED TO IMPLEMENT THIS FUNCTION based on how your Transformer model
    # was designed to accept numerical input.
    # It should take an input dictionary (combining base_inputs and alloy_values)
    # and return a TensorFlow tensor or list of tensors in the format expected by your model.
    def prepare_transformer_input(input_data_dict):
        """
        Prepares the input data for the Keras Transformer model.

        Args:
            input_data_dict (dict): Dictionary containing all input features
                                    (process, initial chemistry, and alloy additions).

        Returns:
            tf.Tensor or list of tf.Tensor: Input data in the format expected
                                          by the Transformer model.
        """
        # Ensure the order of features matches the expected input order of your Transformer model
        # Assuming the order is 'process_chem_cols' followed by 'alloy_cols'
        expected_features = process_chem_cols + alloy_cols
        input_values = [input_data_dict.get(col, 0.0) for col in expected_features]

        # Convert to NumPy array and then to TensorFlow tensor
        input_array = np.array([input_values], dtype=np.float32) # Ensure dtype matches model input

        # *** IMPORTANT: Add any scaling, reshaping, or other transformations
        #               that were applied to the training data before feeding
        #               into the Transformer model. ***
        # Example: If you used a StandardScaler on your training data:
        # scaler = load_your_scaler() # Load your trained scaler
        # input_scaled = scaler.transform(input_array)
        # input_tensor = tf.constant(input_scaled)

        input_tensor = tf.constant(input_array)

        return input_tensor

    # Define the objective function for Bayesian Optimization
    def objective_function(alloy_values):
        input_dict = base_inputs.copy()
        input_dict.update(dict(zip(alloy_cols, alloy_values)))

        # Prepare input using the Transformer-specific preparation function
        transformer_input = prepare_transformer_input(input_dict)

        # Make predictions using the Keras Transformer model
        # Use verbose=0 to suppress prediction progress bar
        prediction = model_transformer.predict(transformer_input, verbose=0)[0]

        # Calculate the objective (e.g., RMSE) against max_chem
        # Ensure prediction and target have the same number of elements for comparison
        # Align target chemistry with the order of the model's output (assuming target_cols order)
        target = np.array([max_chem.get(col, 0) for col in target_cols])

        num_elements_to_compare = min(len(prediction), len(target))
        prediction = prediction[:num_elements_to_compare]
        target = target[:num_elements_to_compare]

        rmse = np.sqrt(np.mean((prediction - target) ** 2))
        return rmse

    # Define the search space (bounds for alloy elements)
    # Using the min/max from the successful heats dataframe
    space = [(df_successful[el].min(), df_successful[el].max()) for el in alloy_cols]

    # Check and adjust bounds (ensure lower bound is less than upper bound)
    space = [(lower, upper) if lower < upper else (lower, lower + 1e-6) for lower, upper in space]

    # Perform Bayesian Optimization
    print("Starting Bayesian Optimization (Transformer Model)...")
    result = gp_minimize(objective_function, space, n_calls=n_calls, random_state=42)

    # Get the optimal alloy additions
    optimized_alloys = dict(zip(alloy_cols, result.x))

    # Make a prediction with the optimized alloys to see the resulting chemistry
    optimized_input_dict = base_inputs.copy()
    optimized_input_dict.update(optimized_alloys)

    # Prepare optimized input for the Transformer model
    optimized_transformer_input = prepare_transformer_input(optimized_input_dict)

    predicted_chem = model_transformer.predict(optimized_transformer_input, verbose=0)[0] # Added verbose=0

    # Compare predicted vs Target (Max), Aim, and Min values
    max_targets = np.array([max_chem.get(col, 0) for col in target_cols])
    aim_targets = np.array([aim_chem.get(col, 0) for col in target_cols])
    min_targets = np.array([min_chem.get(col, 0) for col in target_cols])

    comparison_df = pd.DataFrame({
        "Min Target": min_targets,
        "Aim Target": aim_targets,
        "Max Target": max_targets,
        "Predicted (Optimized)": predicted_chem[:len(target_cols)] # Slice predicted_chem
    }, index=target_cols)


    return optimized_alloys, predicted_chem, comparison_df

# --- How to use the function (assuming necessary variables are already defined) ---

# Example usage (assuming the following variables are available from previous cells):
# filepath = "FE Alloying.xlsx"
#
# # Load necessary data and model (if not already loaded)
# # df, summary_df = preprocess_pipeline(filepath)
# # df_successful = ... # logic to get successful heats
# # base_inputs = ... # logic to get base inputs
# # max_chem = ... # logic to get max target chemistry
# # aim_chem = ... # logic to get aim target chemistry
# # min_chem = ... # logic to get min target chemistry
# # alloy_cols = [...] # list of alloy columns
# # process_chem_cols = [...] # list of process and initial chemistry columns
# # target_cols = [...] # list of target final chemistry columns
# # try:
# #     model_transformer = tf.keras.models.load_model('my_fcn_model.keras')
# # except Exception as e:
# #     print(f"Error loading Keras Transformer model: {e}")
# #     model_transformer = None # Handle case where model is not loaded

# # Assuming model_transformer and other variables are now defined:
# if 'model_transformer' in locals() and model_transformer is not None:
#      optimized_alloys_bo_tf, predicted_chem_bo_tf, comparison_df_bo_tf = run_bayesian_optimization_transformer(
#          model_transformer=model_transformer,
#          df_successful=df_successful, # Pass your df_successful DataFrame
#          base_inputs=base_inputs,     # Pass your base_inputs dictionary
#          max_chem=max_chem,         # Pass your max_chem dictionary
#          aim_chem=aim_chem,         # Pass your aim_chem dictionary
#          min_chem=min_chem,         # Pass your min_chem dictionary
#          alloy_cols=alloy_cols,       # Pass your alloy_cols list
#          process_chem_cols=process_chem_cols, # Pass your process_chem_cols list
#          target_cols=target_cols,     # Pass your target_cols list
#          n_calls=50 # You can adjust the number of optimization calls
#      )

#      if optimized_alloys_bo_tf is not None:
#          print("\nOptimized Alloy Additions (Bayesian Optimization with Transformer):")
#          display(pd.DataFrame(optimized_alloys_bo_tf, index=["Recommended (kg)"]).T.round(4))

#          print("\nPredicted Final Chemistry (with Optimized Alloys) vs Targets:")
#          display(comparison_df_bo_tf)
# else:
#     print("Transformer model not loaded. Cannot run Bayesian Optimization with Transformer.")

In [ ]:
# Install pyswarms if you haven't already
!pip install pyswarms

import numpy as np
import pandas as pd
from pyswarms.single.global_best import GlobalBestPSO  # Import PSO from pyswarms
import tensorflow as tf  # Import tensorflow to load the .keras model
from sklearn.preprocessing import StandardScaler # Import StandardScaler if you used it for your model
import os # Import os for path joining
import datetime # Import datetime for handling time objects

# Assume 'filepath' is defined and points to your data file
# Assume 'scaler' used to standardize the features for the FCN model is available
# Assume 'model_dir' is defined and is the directory where your model was saved
# Assume the model file is named 'my_fcn_model.keras'

# --- Reference Data Loading and Preprocessing ---
# (Keep the same preprocessing steps as used before training your model)
# Make sure preprocess_pipeline is defined and accessible
def load_data(filepath):
    import os # Import os locally
    df = pd.read_excel(filepath, sheet_name="Heats") if os.path.basename(filepath).endswith('.xlsx') else pd.read_csv(filepath)
    df.columns = df.columns.str.strip()
    if df.iloc[0].isnull().sum() < 5:
        df.columns = df.iloc[0]
        df = df[1:]
        df.columns = df.columns.str.strip()
    df.dropna(axis=1, how='all', inplace=True)
    df.dropna(axis=0, how='all', inplace=True)
    return df

def load_summary(filepath):
    import os # Import os locally
    if os.path.basename(filepath).endswith('.xlsx'):
        summary_df = pd.read_excel(filepath, sheet_name="Summary")
        summary_df.columns = summary_df.columns.str.strip()
        summary_df = summary_df.dropna(how='all')
        return summary_df
    return None

def handle_missing(df):
    df.fillna(method='ffill', inplace=True)
    df.fillna(df.median(numeric_only=True), inplace=True)
    return df

def create_delta_columns(df):
    open_chem = ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']
    final_chem = [f"F-{el}" for el in ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']]
    for open_col, final_col in zip(open_chem, final_chem):
        if open_col in df.columns and final_col in df.columns:
            delta_col = f"Delta_{open_col.replace('%', '')}"
            df[delta_col] = df[final_col] - df[open_col]
    return df

def clip_outliers(df):
    import numpy as np # Import numpy locally
    for col in df.select_dtypes(include=[np.number]).columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        df[col] = df[col].clip(lower=q1 - 1.5 * iqr, upper=q3 + 1.5 * iqr)
    return df


def preprocess_pipeline(filepath):
    df = load_data(filepath)
    summary_df = load_summary(filepath)
    df = handle_missing(df)
    df = create_delta_columns(df)
    df = clip_outliers(df)
    return df, summary_df

# Re-create and fit the scaler using the exact same features and data split as model training
# Assuming the previous split was 80% train, 20% test
df_temp, summary_df_temp = preprocess_pipeline(filepath) # Use temp variables to not overwrite main df/summary_df
alloy_cols_temp = [
    "CSP-SiMn", "Mn HC", "Mn MC", "Mn LC", "Mn Metal", "FeSi", "Ladle Cov",
    "FeMo Metal", "FeV", "FeNb lumps", "FeTi lumps", "FeTi Wire", "FeB", "FeAl",
    "Cal Carb", "Al bar", "Al  wire", "FeP", "Sul Stick", "Al mix", "CaSi wire",
    "Cal Wire", "CaFeAl Wire", "S Wire", "Ni Plate", "FeCr LC", "FeCr HC",
    "Al Shot", "Lead Wire", "Mo Metal", "Syn Slag"
]
process_chem_cols_temp = [
    'Lift Temp', 'Liquidus temp (° C)', 'Arching Time-mm',
    'LRF Holding Time-mm', 'LRF Lime',
    'C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%'
]
features_temp = alloy_cols_temp + process_chem_cols_temp
target_cols_temp = [f"F-{el}" for el in ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']]

X_temp = df_temp[features_temp]
y_temp = df_temp[target_cols_temp]

X_train_temp, X_test_temp, y_train_temp, y_test_temp = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

# Convert datetime.time columns in the training data (if any)
for col in X_train_temp.select_dtypes(include=['object']).columns:
    if X_train_temp[col].apply(lambda x: isinstance(x, (datetime.time))).any():
        # Convert datetime.time to a numerical format (e.g., total seconds)
        X_train_temp[col] = pd.to_numeric(X_train_temp[col].astype(str).str.replace(':', ''), errors='coerce').fillna(0)


scaler = StandardScaler()
scaler.fit(X_train_temp) # Fit the scaler on the training data

# Now continue with the original df for optimization
df, summary_df = preprocess_pipeline(filepath)

# Define alloy and process features (same as used for training)
alloy_cols = [
    "CSP-SiMn", "Mn HC", "Mn MC", "Mn LC", "Mn Metal", "FeSi", "Ladle Cov",
    "FeMo Metal", "FeV", "FeNb lumps", "FeTi lumps", "FeTi Wire", "FeB", "FeAl",
    "Cal Carb", "Al bar", "Al  wire", "FeP", "Sul Stick", "Al mix", "CaSi wire",
    "Cal Wire", "CaFeAl Wire", "S Wire", "Ni Plate", "FeCr LC", "FeCr HC",
    "Al Shot", "Lead Wire", "Mo Metal", "Syn Slag"
]
process_chem_cols = [
    'Lift Temp', 'Liquidus temp (° C)', 'Arching Time-mm',
    'LRF Holding Time-mm', 'LRF Lime',
    'C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%'
]
target_cols = [f"F-{el}" for el in ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']]

# Select features and target (using the same columns as model training)
features = alloy_cols + process_chem_cols
target = target_cols

# Handle potential missing columns
features = [col for col in features if col in df.columns]
target = [col for col in target if col in df.columns]

# You'll need the original DataFrame 'df' here for determining bounds based on successful heats
# and for creating the base_inputs

# Extract successful heats
max_summary_row = summary_df.iloc[2].fillna(0)
aim_summary_row = summary_df.iloc[3].fillna(0)
min_summary_row = summary_df.iloc[1].fillna(0)
print(aim_summary_row)
target_keys = [k.strip() for k in max_summary_row.index if k.strip().endswith("%")]
target_vector = max_summary_row[target_keys].dropna().astype(float)
present_f_cols = [f"F-{k}" for k in target_keys if f"F-{k}" in df.columns]
df['Success_Score'] = -((df[present_f_cols] - target_vector.loc[[k for k in target_keys if f"F-{k}" in df.columns]].values) ** 2).sum(axis=1) ** 0.3
df_successful = df[df['Success_Score'] >= df['Success_Score'].quantile(0.5)]

print(df_successful.shape)  # Check if it's empty
print(df_successful[alloy_cols].describe())  # Check summary statistics

# Select base inputs from median of successful heats (using the same feature columns as the model)
selected_cols = [col for col in process_chem_cols + alloy_cols if col in df_successful.columns and col in features]
base_inputs = df_successful[selected_cols].median(numeric_only=True).to_dict()

# Extract target chemistry (using the same target columns as the model)
max_chem = {
    f"F-{k.strip()}": float(v) for k, v in max_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v) and f"F-{k.strip()}" in target
}

aim_chem = {
    f"F-{k.strip()}": float(v) for k, v in aim_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v) and f"F-{k.strip()}" in target
}

min_chem = {
    f"F-{k.strip()}": float(v) for k, v in min_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v) and f"F-{k.strip()}" in target
}

# --- Load the Keras model ---
model_dir = 'models'
model_path = os.path.join(model_dir, 'my_fcn_model.keras')
try:
    model = tf.keras.models.load_model(model_path)
    print(f"Model loaded successfully from {model_path}")
except Exception as e:
    print(f"Error loading model: {e}")
    model = None # Set model to None if loading fails

# --- PSO Implementation using the loaded Keras model ---

if model is not None:
    # Define the optimization function for PSO using the loaded model
    def run_pso_optimization(model, base_inputs, target_chemistry, alloy_elements, df_successful, scaler, all_features):
        # Bounds for each alloy based on min/max scaling in successful heats
        pso_alloy_elements = [el for el in alloy_elements if el in all_features]
        bounds = [(df_successful[el].min(), df_successful[el].max()) for el in pso_alloy_elements]

        # Display bounds inline
        print("Alloy Bounds for PSO:")
        for i, el in enumerate(pso_alloy_elements):
            print(f"{el}: Min = {bounds[i][0]:.2f}, Max = {bounds[i][1]:.4f}")

        # Prepare input for the model, including scaling
        def prepare_input(alloy_values):
            input_dict = base_inputs.copy()
            input_dict.update(dict(zip(pso_alloy_elements, alloy_values)))

            # Create a DataFrame with all the model's expected features
            input_df = pd.DataFrame([input_dict])

            # ***FIX***: Reindex to the exact feature order used during scaler fitting
            input_df = input_df.reindex(columns=all_features, fill_value=0)

            # Convert datetime.time objects if any (based on your preprocessing)
            # Ensure this matches how datetime.time was handled when fitting the scaler
            for col in input_df.select_dtypes(include=['object']).columns:
                 if input_df[col].apply(lambda x: isinstance(x, datetime.time)).any():
                    # Example conversion (adjust if your initial preprocessing was different)
                    input_df[col] = pd.to_numeric(input_df[col].astype(str).str.replace(':', ''), errors='coerce').fillna(0)
                 else:
                     input_df[col] = pd.to_numeric(input_df[col], errors='coerce').fillna(0)


            # Scale the input using the fitted scaler
            input_scaled = scaler.transform(input_df)

            return input_scaled

        # Fitness function for PSO
        def fitness_function(alloy_matrix):
            num_particles = alloy_matrix.shape[0]
            losses = []

            for i in range(num_particles):
                alloy_values = alloy_matrix[i]
                input_scaled = prepare_input(alloy_values)

                # Predict using the loaded Keras model
                prediction = model.predict(input_scaled, verbose=0)[0]

                # Ensure target chemistry keys align with the model's prediction output
                # Assuming the order of target_cols matches the order of model outputs
                ordered_target_values = np.array([target_chemistry.get(col, 0) for col in target])

                # Calculate loss (RMSE) between prediction and target
                num_elements_to_compare = min(len(prediction), len(ordered_target_values))
                prediction_subset = prediction[:num_elements_to_compare]
                target_subset = ordered_target_values[:num_elements_to_compare]

                loss = np.sqrt(np.mean((prediction_subset - target_subset) ** 2))
                losses.append(loss)

            return np.array(losses)

        # Define the bounds for the alloy elements included in the PSO
        lower_bounds = [bound[0] for bound in bounds]
        upper_bounds = [bound[1] for bound in bounds]
        n_particles = 50  # Adjust as needed

        # Initialize and run PSO
        options = {'c1': 0.5, 'c2': 0.7, 'w': 0.4}  # Adjust parameters as needed
        optimizer = GlobalBestPSO(n_particles=n_particles, dimensions=len(pso_alloy_elements),
                                options=options, bounds=(lower_bounds, upper_bounds))
        cost, pos = optimizer.optimize(fitness_function, iters=100)  # Adjust iterations as needed

        # Best solution and prediction
        best_individual_alloy_values = pos  # The best particle's position (alloy values)
        best_individual_inputs_scaled = prepare_input(best_individual_alloy_values)
        best_prediction = model.predict(best_individual_inputs_scaled, verbose=0)[0]

        # Return the optimized alloys and predicted chemistry
        optimized_alloy_additions = dict(zip(pso_alloy_elements, best_individual_alloy_values))
        return optimized_alloy_additions, best_prediction

    # Run PSO optimization using the loaded model and scaler
    # Ensure `scaler` is fitted on the training data (X_train)
    # Ensure `features` is the list of column names used to train the model
    optimized_alloys_pso, predicted_chem_pso = run_pso_optimization(
        model=model,
        base_inputs=base_inputs,
        target_chemistry=max_chem,  # Or use aim_chem
        alloy_elements=alloy_cols,
        df_successful=df_successful,
        scaler=scaler, # Pass the fitted scaler
        all_features=features # Pass the list of all features used by the model
    )

    # Display results
    print("\nOptimized Alloy Additions (PSO)")
    display_optimized_alloys = {col: optimized_alloys_pso.get(col, 0.0) for col in alloy_cols}
    print(pd.DataFrame(display_optimized_alloys, index=["Recommended (kg)"]))

    print("\nPredicted Final Chemistry vs Target (PSO)")
    # Ensure the keys for the 'Predicted' column match the target columns predicted by the model
    # Assuming the order of model outputs aligns with the order of 'target' list
    predicted_chem_dict = dict(zip([col for col in target if col in max_chem or col in aim_chem or col in min_chem], predicted_chem_pso[:len([col for col in target if col in max_chem or col in aim_chem or col in min_chem])]))

    chem_df_pso = pd.DataFrame({
        "Min": pd.Series(min_chem),
        "Target": pd.Series(max_chem),
        "Aim": pd.Series(aim_chem),
        "Predicted": predicted_chem_dict # Use the created dictionary
    })

    print(chem_df_pso.round(5))
else:
    print("Model was not loaded. Skipping PSO optimization.")

<ipython-input-105-ee58cf70cdf5>:42: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
<ipython-input-105-ee58cf70cdf5>:42: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(method='ffill', inplace=True)
<ipython-input-105-ee58cf70cdf5>:42: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
<ipython-input-105-ee58cf70cdf5>:42: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the f

Unnamed: 0                   0
Unnamed: 1                   0
Steel Grade > G-600-A      Aim
C%                        0.05
Mn%                       0.12
S%                         0.0
P%                       0.015
Si%                       0.04
Cr%                        0.0
Ni%                        0.0
Mo%                        0.0
V%                         0.0
Ti%                        0.0
Al%                      0.038
Ca%                        0.0
N%                         0.0
Pb%                        0.0
Nb%                        0.0
B%                         0.0
CU%                        0.0
Name: 3, dtype: object
(1621, 89)
0      CSP-SiMn        Mn HC   Mn MC   Mn LC  Mn Metal    FeSi  Ladle Cov  \
count    1621.0  1621.000000  1621.0  1621.0    1621.0  1621.0     1621.0   
mean        0.0    15.586058     0.0     0.0       0.0     0.0        0.0   
std         0.0    23.905463     0.0     0.0       0.0     0.0        0.0   
min         0.0     0.000000     0.0  

pyswarms.single.global_best:   0%|          |0/100, best_cost=0.0932/usr/local/lib/python3.11/dist-packages/pyswarms/backend/handlers.py:387: RuntimeWarning: invalid value encountered in remainder
  new_pos[greater_than_bound] = lb[greater_than_bound] + np.mod(
pyswarms.single.global_best: 100%|██████████|100/100, best_cost=0.0932
2025-05-18 13:25:44,587 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.09316503409733069, best pos: [  0.          20.19611703   0.           0.           0.
   0.           0.           0.           0.           0.
   0.           0.           0.           0.           0.
   0.         165.6315682    0.           0.           0.
   0.          18.52115819   0.           0.           0.
   0.           0.           0.           0.           0.
   0.        ]



Optimized Alloy Additions (PSO)
                  CSP-SiMn      Mn HC  Mn MC  Mn LC  Mn Metal  FeSi  \
Recommended (kg)       0.0  20.196117    0.0    0.0       0.0   0.0   

                  Ladle Cov  FeMo Metal  FeV  FeNb lumps  ...   Cal Wire  \
Recommended (kg)        0.0         0.0  0.0         0.0  ...  18.521158   

                  CaFeAl Wire  S Wire  Ni Plate  FeCr LC  FeCr HC  Al Shot  \
Recommended (kg)          0.0     0.0       0.0      0.0      0.0      0.0   

                  Lead Wire  Mo Metal  Syn Slag  
Recommended (kg)        0.0       0.0       0.0  

[1 rows x 31 columns]

Predicted Final Chemistry vs Target (PSO)
        Min  Target    Aim  Predicted
F-C%   0.02   0.060  0.050    0.11365
F-Mn%  0.10   0.150  0.120   -0.05939
F-S%   0.00   0.010  0.000    0.00624
F-P%   0.00   0.030  0.015   -0.06230
F-Si%  0.00   0.050  0.040    0.12664
F-Cr%  0.00   0.050  0.000    0.05810
F-Ni%  0.00   0.050  0.000   -0.10243
F-Mo%  0.00   0.005  0.000    0.05773
F-V%  

In [ ]:

# Load and preprocess data
df, summary_df = preprocess_pipeline(filepath)

# Define features and targets
alloy_cols = [
    "CSP-SiMn", "Mn HC", "Mn MC", "Mn LC", "Mn Metal", "FeSi", "Ladle Cov",
    "FeMo Metal", "FeV", "FeNb lumps", "FeTi lumps", "FeTi Wire", "FeB", "FeAl",
    "Cal Carb", "Al bar", "Al  wire", "FeP", "Sul Stick", "Al mix", "CaSi wire",
    "Cal Wire", "CaFeAl Wire", "S Wire", "Ni Plate", "FeCr LC", "FeCr HC",
    "Al Shot", "Lead Wire", "Mo Metal", "Syn Slag"
]
process_chem_cols = [
    'Lift Temp', 'Liquidus temp (° C)', 'Arching Time-mm',
    'LRF Holding Time-mm', 'LRF Lime',
    'C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%'
]
target_cols = [f"F-{el}" for el in ['C%', 'Mn%', 'S%', 'P%', 'Si%', 'Cr%', 'Ni%', 'Mo%', 'V%', 'Ti%', 'Al%', 'Ca%', 'N%', 'Pb%', 'Nb%']]


# Extract successful heats
max_summary_row = summary_df.iloc[2].fillna(0)
aim_summary_row = summary_df.iloc[3].fillna(0)
min_summary_row = summary_df.iloc[1].fillna(0)

target_keys = [k.strip() for k in max_summary_row.index if k.strip().endswith("%")]
target_vector = max_summary_row[target_keys].dropna().astype(float)
present_f_cols = [f"F-{k}" for k in target_keys if f"F-{k}" in df.columns]
df['Success_Score'] = -((df[present_f_cols] - target_vector.loc[[k for k in target_keys if f"F-{k}" in df.columns]].values) ** 2).sum(axis=1) ** 0.3
df_successful = df[df['Success_Score'] >= df['Success_Score'].quantile(0.5)]

# Select base inputs from median of successful heats
selected_cols = [col for col in process_chem_cols + alloy_cols if col in df_successful.columns]
base_inputs = df_successful[selected_cols].median(numeric_only=True).to_dict()

# Extract target chemistry dictionaries
max_chem = {
    f"F-{k.strip()}": float(v) for k, v in max_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v)
}

aim_chem = {
    f"F-{k.strip()}": float(v) for k, v in aim_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v)
}

min_chem = {
    f"F-{k.strip()}": float(v) for k, v in min_summary_row.items()
    if k.strip().endswith("%") and pd.notnull(v)
}


# Load the Keras Transformer model
model_path_keras = os.path.join('models', 'my_fcn_model.keras')
try:
    model_transformer = tf.keras.models.load_model(model_path_keras)
    print(f"Keras Transformer model loaded successfully from {model_path_keras}")
except Exception as e:
    print(f"Error loading Keras Transformer model: {e}")
    print(f"Please ensure '{model_path_keras}' exists and is in the correct directory.")
    model_transformer = None # Set to None if loading fails


# --- Call the Bayesian Optimization function (Assuming model is loaded) ---

if model_transformer is not None:
     optimized_alloys_bo_tf, predicted_chem_bo_tf, comparison_df_bo_tf = run_bayesian_optimization_transformer(
         model_transformer=model_transformer,
         df_successful=df_successful, # Pass your df_successful DataFrame
         base_inputs=base_inputs,     # Pass your base_inputs dictionary
         max_chem=max_chem,         # Pass your max_chem dictionary
         aim_chem=aim_chem,         # Pass your aim_chem dictionary
         min_chem=min_chem,         # Pass your min_chem dictionary
         alloy_cols=alloy_cols,       # Pass your alloy_cols list
         process_chem_cols=process_chem_cols, # Pass your process_chem_cols list
         target_cols=target_cols,     # Pass your target_cols list
         n_calls=50 # You can adjust the number of optimization calls
     )

     if optimized_alloys_bo_tf is not None:
         print("\nOptimized Alloy Additions (Bayesian Optimization with Transformer):")
         display(pd.DataFrame(optimized_alloys_bo_tf, index=["Recommended (kg)"]).T.round(4))

         print("\nPredicted Final Chemistry (with Optimized Alloys) vs Targets:")
         display(comparison_df_bo_tf)
else:
    print("Transformer model not loaded. Cannot run Bayesian Optimization.")

<ipython-input-105-ee58cf70cdf5>:42: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)
<ipython-input-105-ee58cf70cdf5>:42: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(method='ffill', inplace=True)


Keras Transformer model loaded successfully from models/my_fcn_model.keras
Starting Bayesian Optimization (Transformer Model)...

Optimized Alloy Additions (Bayesian Optimization with Transformer):


,Recommended (kg)
CSP-SiMn,0.00
Mn HC,80.00
Mn MC,0.00
Mn LC,0.00
Mn Metal,0.00
FeSi,0.00
Ladle Cov,0.00
FeMo Metal,0.00
FeV,0.00
FeNb lumps,0.00



Predicted Final Chemistry (with Optimized Alloys) vs Targets:


,Min Target,Aim Target,Max Target,Predicted (Optimized)
F-C%,0.02,0.050,0.060,53.445232
F-Mn%,0.10,0.120,0.150,-277.938263
F-S%,0.00,0.000,0.010,34.836517
F-P%,0.00,0.015,0.030,-138.050201
F-Si%,0.00,0.040,0.050,190.193878
F-Cr%,0.00,0.000,0.050,218.871521
F-Ni%,0.00,0.000,0.050,4.679028
F-Mo%,0.00,0.000,0.005,156.512512
F-V%,0.00,0.000,0.003,-25.458565
F-Ti%,0.00,0.000,0.003,131.109558
